In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse variable names across steps for efficiency
tz = pytz.timezone('Asia/Shanghai')

# Load files
df = pd.read_csv('metric.csv')
df2 = pd.read_csv('trace.csv')
df3 = pd.read_csv('log.csv')
df4 = pd.read_csv('error_logs.csv')

# --- metric.csv summaries ---
metric_row_count = df.shape[0]

top_cmdb_metric = (
    df['cmdb_id']
    .value_counts()
    .head(20)
    .rename_axis('cmdb_id')
    .reset_index(name='row_count')
)

kpi_distinct = (
    df.groupby('kpi_name')['cmdb_id']
    .nunique()
    .reset_index(name='distinct_cmdb_count')
)
kpi_occurrence = (
    df.groupby('kpi_name').size().reset_index(name='total_occurrences')
)
kpi_summary = (
    kpi_distinct.merge(kpi_occurrence, on='kpi_name')
    .sort_values(['distinct_cmdb_count', 'total_occurrences'], ascending=[False, False])
    .reset_index(drop=True)
    .head(50)  # limit to top 50 kpi_names by occurrence/coverage
)

# --- trace.csv summaries ---
trace_row_count = df2.shape[0]

top_cmdb_trace = (
    df2['cmdb_id']
    .value_counts()
    .head(20)
    .rename_axis('cmdb_id')
    .reset_index(name='row_count')
)

trace_distinct = (
    df2.groupby('trace_name')['cmdb_id']
    .nunique()
    .reset_index(name='distinct_cmdb_count')
)
trace_occurrence = (
    df2.groupby('trace_name').size().reset_index(name='total_occurrences')
)
trace_summary = (
    trace_distinct.merge(trace_occurrence, on='trace_name')
    .sort_values(['distinct_cmdb_count', 'total_occurrences'], ascending=[False, False])
    .reset_index(drop=True)
    .head(50)
)

# --- log.csv summaries ---
log_row_count = df3.shape[0]

top_cmdb_log = (
    df3['cmdb_id']
    .value_counts()
    .head(20)
    .rename_axis('cmdb_id')
    .reset_index(name='row_count')
)

log_distinct = (
    df3.groupby('log_name')['cmdb_id']
    .nunique()
    .reset_index(name='distinct_cmdb_count')
)
log_occurrence = (
    df3.groupby('log_name').size().reset_index(name='total_occurrences')
)
log_summary = (
    log_distinct.merge(log_occurrence, on='log_name')
    .sort_values(['distinct_cmdb_count', 'total_occurrences'], ascending=[False, False])
    .reset_index(drop=True)
    .head(50)
)

# --- error_logs.csv summaries ---
error_row_count = df4.shape[0]

top_cmdb_error = (
    df4['cmdb_id']
    .value_counts()
    .head(20)
    .rename_axis('cmdb_id')
    .reset_index(name='row_count')
)

# Earliest and latest timestamp in UTC+8
if 'timestamp' in df4.columns and not df4['timestamp'].isnull().all():
    times_utc = pd.to_datetime(df4['timestamp'], unit='s', utc=True)
    times_local = times_utc.dt.tz_convert(tz)
    earliest_local = times_local.min()
    latest_local = times_local.max()
    earliest_iso = earliest_local.isoformat()
    latest_iso = latest_local.isoformat()
else:
    earliest_iso = None
    latest_iso = None

# Assemble compact summary dict (use DataFrames for tabular parts; limited sizes applied)
summary = {
    'metric.csv': {
        'total_rows': int(metric_row_count),
        'top_20_cmdb_id_counts': top_cmdb_metric,   # DataFrame (cmdb_id, row_count)
        'top_kpi_names_by_distinct_cmdbs': kpi_summary  # DataFrame (kpi_name, distinct_cmdb_count, total_occurrences)
    },
    'trace.csv': {
        'total_rows': int(trace_row_count),
        'top_20_cmdb_id_counts': top_cmdb_trace,
        'top_trace_names_by_distinct_cmdbs': trace_summary
    },
    'log.csv': {
        'total_rows': int(log_row_count),
        'top_20_cmdb_id_counts': top_cmdb_log,
        'top_log_names_by_distinct_cmdbs': log_summary
    },
    'error_logs.csv': {
        'total_rows': int(error_row_count),
        'top_20_cmdb_id_counts': top_cmdb_error,
        'earliest_timestamp_UTC+8': earliest_iso,
        'latest_timestamp_UTC+8': latest_iso
    }
}

# Display the compact summary variable (IPython will render DataFrames inside the dict compactly)
summary

```
Out[1]:
```
```python
# Summary string based on the executed summaries (compact, plain English)
summary_text = (
    "Summary of telemetry files:\n\n"
    "metric.csv: total rows = 149,636. Top cmdb_ids by row count (top examples): "
    "adservice (10,230), adservice2 (9,990), frontend2-0 (5,224), frontend-0 (5,160), "
    "frontend-1 (5,160), frontend-2 (4,980). The top KPI names (limited to 50) are "
    "dominated by container.node-5.* metrics (network/memory-related), many appearing "
    "across ~33–35 distinct cmdb_ids with high occurrence counts.\n\n"
    "trace.csv: total rows = 36,232. Top cmdb_ids: frontend-0 (2,752), frontend-1 (2,728), "
    "frontend-2 (2,688), checkoutservice-1 (2,652), checkoutservice-2 (2,644). Top trace names "
    "include trace.from_frontend-0.duration_mean, duration_p95, error_rate, row_count (each "
    "appearing across ~21 cmdb_ids with ~628 occurrences for those top entries).\n\n"
    "log.csv: total rows = 1,814. Top cmdb_ids (many equal counts) include adservice-0/1/2, "
    "cartservice-0/1/2, checkoutservice-0/1/2, frontend-0/1/2, etc., typically 60 rows each. "
    "Top log names: log.error_count and log.row_count (each appearing in 31 distinct cmdb_ids, "
    "~907 occurrences).\n\n"
    "error_logs.csv: total rows = 1,697. Top cmdb_ids by error count: frontend-0 (772), "
    "frontend-1 (535), frontend-2 (390). Earliest timestamp (UTC+8): 2022-03-21T16:30:01+08:00. "
    "Latest timestamp (UTC+8): 2022-03-21T16:59:59+08:00.\n\n"
    "If you want more detail on any of these (e.g., full top-50 KPI/trace/log lists or more cmdb rows), "
    "ask and I will display the requested subset."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 149636, 'top_20_cmdb_id_counts':                      cmdb_id  row_count
0                  adservice      10230
1                 adservice2       9990
2                frontend2-0       5224
3                 frontend-0       5160
4                 frontend-1       5160
5                 frontend-2       4980
6          checkoutservice-0       4020
7          checkoutservice-2       4020
8          checkoutservice-1       3900
9         checkoutservice2-0       3540
10   productcatalogservice-1       3480
11   productcatalogservice-2       3480
12   productcatalogservice-0       3480
13             cartservice-1       3000
14              adservice2-0       2956
15             cartservice-0       2880
16  productcatalogservice2-0       2824
17            cartservice2-0       2824
18               adservice-0       2700
19               adservice-2       2700, 'top_kpi_names_by_distinct_cmdbs':                                              kpi_name  distinct_cmdb_count  total_occurrences
0   container.node-5.container_network_receive_MB....                   35               1050
1   container.node-5.container_network_receive_err...                   35               1050
2   container.node-5.container_network_receive_pac...                   35               1050
3   container.node-5.container_network_receive_pac...                   35               1050
4   container.node-5.container_network_transmit_MB...                   35               1050
..                                                ...                  ...                ...
45             container.node-5.container_memory_swap                   33                990
46         container.node-5.container_memory_usage_MB                   33                990
47   container.node-5.container_memory_working_set_MB                   33                990
48                 container.node-5.container_sockets                   33                990
49         container.node-5.container_spec_cpu_period                   33                990

[50 rows x 3 columns]}, 'trace.csv': {'total_rows': 36232, 'top_20_cmdb_id_counts':                     cmdb_id  row_count
0                frontend-0       2752
1                frontend-1       2728
2                frontend-2       2688
3         checkoutservice-1       2652
4         checkoutservice-2       2644
5         checkoutservice-0       2620
6   productcatalogservice-1       1200
7   productcatalogservice-0       1196
8   productcatalogservice-2       1188
9               frontend2-0       1080
10       checkoutservice2-0       1080
11  recommendationservice-0        840
12  recommendationservice-2        840
13  recommendationservice-1        840
14            cartservice-2        840
15            cartservice-1        836
16            cartservice-0        832
17        shippingservice-2        720
18        currencyservice-2        720
19        currencyservice-0        720, 'top_trace_names_by_distinct_cmdbs':                                      trace_name  distinct_cmdb_count  total_occurrences
0           trace.from_frontend-0.duration_mean                   21                628
1            trace.from_frontend-0.duration_p95                   21                628
2              trace.from_frontend-0.error_rate                   21                628
3               trace.from_frontend-0.row_count                   21                628
4           trace.from_frontend-1.duration_mean                   21                622
..                                          ...                  ...                ...
45          trace.from_frontend2-0.duration_p95                    7                210
46            trace.from_frontend2-0.error_rate                    7                210
47             trace.from_frontend2-0.row_count                    7                210
48  trace.from_checkoutservice2-0.duration_mean                    6                180
49   trace.from_checkoutservice2-0.duration_p95                    6                180

[50 rows x 3 columns]}, 'log.csv': {'total_rows': 1814, 'top_20_cmdb_id_counts':               cmdb_id  row_count
0         adservice-0         60
1         adservice-1         60
2         adservice-2         60
3       cartservice-0         60
4       cartservice-1         60
5       cartservice-2         60
6   checkoutservice-0         60
7   checkoutservice-1         60
8   checkoutservice-2         60
9   currencyservice-0         60
10  currencyservice-1         60
11  currencyservice-2         60
12     emailservice-0         60
13     emailservice-1         60
14     emailservice-2         60
15         frontend-0         60
16         frontend-1         60
17         frontend-2         60
18   paymentservice-0         60
19   paymentservice-1         60, 'top_log_names_by_distinct_cmdbs':           log_name  distinct_cmdb_count  total_occurrences
0  log.error_count                   31                907
1    log.row_count                   31                907}, 'error_logs.csv': {'total_rows': 1697, 'top_20_cmdb_id_counts':       cmdb_id  row_count
0  frontend-0        772
1  frontend-1        535
2  frontend-2        390, 'earliest_timestamp_UTC+8': '2022-03-21T16:30:01+08:00', 'latest_timestamp_UTC+8': '2022-03-21T16:59:59+08:00'}}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing variables: df (metric DataFrame) and tz (Asia/Shanghai) if present.
# If not present (fresh kernel), load metric.csv and set tz.
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using ALL rows (before any filtering)
p95_df = (
    df.groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Convert timestamp to timezone-aware Asia/Shanghai and filter to incident window (inclusive)
df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df_window = df[(df['ts'] >= start_ts) & (df['ts'] <= end_ts)].copy()

#  Merge p95 thresholds into the windowed rows so we can mark anomalies
df_window = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies in the window: value >= group's global p95_threshold
df_window['is_anomaly'] = df_window['value'] >= df_window['p95_threshold']

# 3) For each series compute anomaly_count, earliest anomaly ts in window, and max value in window
# Aggregates from windowed data
if not df_window.empty:
    agg_window_counts = (
        df_window.groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )
    # earliest anomaly timestamp per group (only where is_anomaly True)
    earliest_anom = (
        df_window[df_window['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'])['ts']
        .min()
        .reset_index(name='earliest_anomaly_ts')
    )
else:
    agg_window_counts = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','max_value_in_window'])
    earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_ts'])

# Combine aggregates with full p95_df to include series that had no rows in the window
summary_df = (
    p95_df
    .merge(agg_window_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing values: series with no rows in window -> anomaly_count = 0, max_value_in_window = NaN, earliest_anomaly_ts = NaT
summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_ts already NaT where missing; max_value_in_window can stay NaN
# Convert earliest_anomaly_ts to ISO strings (UTC+8) or None
if 'earliest_anomaly_ts' in summary_df.columns:
    summary_df['earliest_anomaly_timestamp'] = summary_df['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    # Convert empty strings for NaT to None and adjust format to include colon in timezone offset
    def iso_or_none(s):
        if pd.isna(s):
            return None
        # format like 2022-03-21T16:30:01+0800 -> insert colon -> +08:00
        return s[:-5] + s[-5:-2] + ':' + s[-2:]
    summary_df['earliest_anomaly_timestamp'] = summary_df['earliest_anomaly_timestamp'].apply(iso_or_none)
else:
    summary_df['earliest_anomaly_timestamp'] = None

# Keep only requested columns and order
summary_df = summary_df[[
    'cmdb_id',
    'kpi_name',
    'p95_threshold',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window'
]]

# 4) Build compact outputs
total_series_scanned = int(p95_df.shape[0])
num_series_with_anomalies = int((summary_df['anomaly_count'] > 0).sum())

# Top 20 series by anomaly_count desc, tie-breaker by max_value_in_window desc
top_20 = (
    summary_df
    .sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Assemble final compact dict-like summary
result_summary = {
    'total_series_scanned': total_series_scanned,
    'series_with_>=1_anomalies_in_window': num_series_with_anomalies,
    'top_20_by_anomaly_count': top_20  # DataFrame (limited to 20 rows)
}

# Display the compact summary variable (IPython will render the top_20 DataFrame compactly)
result_summary

```
Out[2]:
```
```python
# Summarize the previously computed result_summary (from metric.csv analysis) in plain English.
try:
    result_summary
except NameError:
    raise RuntimeError("result_summary not found in kernel. Please run the analysis cell first.")

# Extract key numbers and an example top-20 snippet for concise reporting
total_series = result_summary['total_series_scanned']
series_with_anoms = result_summary['series_with_>=1_anomalies_in_window']
top20 = result_summary['top_20_by_anomaly_count']

# Build a compact plain-English summary
summary_text = (
    f"Metric.csv incident-window anomaly summary:\n\n"
    f"- Total distinct (cmdb_id, kpi_name) series scanned: {total_series}\n"
    f"- Series with >=1 anomaly in the window (value >= global P95): {series_with_anoms}\n\n"
    f"Key observations from the top 20 series by anomaly count:\n"
    f"- All top entries have anomaly_count = 30 and earliest anomaly timestamp = 2022-03-21T16:30:00+08:00.\n"
    f"- Representative top series (cmdb_id : kpi_name) include:\n"
    f"  * adservice2 : runtime.java_lang_Runtime_StartTime\n"
    f"  * adservice  : runtime.java_lang_Runtime_StartTime\n"
    f"  * node-4/node-1/node-2/node-3/node-6 : system.disk.total\n"
    f"  * several services (cartservice*, productcatalogservice*, currencyservice*, paymentservice*, recommendationservice*, frontend2-0) : container.node-6.container_start_time_seconds\n\n"
    f"Interpretation (data-only):\n"
    f"- Every scanned series (all {total_series}) had at least one value at-or-above its global P95 during 2022-03-21T16:30:00+08:00 through 2022-03-21T17:00:00+08:00.\n"
    f"- The top series show constant/peak values at the start of the window (earliest anomaly = 16:30:00+08:00) and identical anomaly counts (30), suggesting the window captured sustained values >= historical P95 for these KPIs.\n\n"
    f"If you want the exact top-20 table (columns: cmdb_id, kpi_name, p95_threshold, anomaly_count, earliest_anomaly_timestamp, max_value_in_window),\n"
    f"inspect the 'top_20_by_anomaly_count' DataFrame: top20.head(20)."
)

# Display the summary string (compact, plain English)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_series_scanned': 5014, 'series_with_>=1_anomalies_in_window': 5014, 'top_20_by_anomaly_count':                      cmdb_id                                           kpi_name  p95_threshold  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0                 adservice2                runtime.java_lang_Runtime_StartTime   1.647822e+12             30  2022-03-21T16:30:00+08:00         1.647822e+12
1                  adservice                runtime.java_lang_Runtime_StartTime   1.647523e+12             30  2022-03-21T16:30:00+08:00         1.647523e+12
2                     node-4                                  system.disk.total   7.616300e+09             30  2022-03-21T16:30:00+08:00         7.616300e+09
3                     node-1                                  system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
4                     node-2                                  system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
5                     node-3                                  system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
6                     node-6                                  system.disk.total   3.617924e+09             30  2022-03-21T16:30:00+08:00         3.617924e+09
7                 adservice2  runtime.java_lang_OperatingSystem_CommittedVir...   3.566666e+09             30  2022-03-21T16:30:00+08:00         3.566666e+09
8                  adservice  runtime.java_lang_OperatingSystem_CommittedVir...   3.387429e+09             30  2022-03-21T16:30:00+08:00         3.393109e+09
9             cartservice2-0      container.node-6.container_start_time_seconds   1.647841e+09             30  2022-03-21T16:30:00+08:00         1.647841e+09
10              adservice2-0      container.node-6.container_start_time_seconds   1.647822e+09             30  2022-03-21T16:30:00+08:00         1.647822e+09
11             cartservice-0      container.node-5.container_start_time_seconds   1.647778e+09             30  2022-03-21T16:30:00+08:00         1.647778e+09
12             cartservice-1      container.node-5.container_start_time_seconds   1.647778e+09             30  2022-03-21T16:30:00+08:00         1.647778e+09
13             cartservice-2      container.node-5.container_start_time_seconds   1.647778e+09             30  2022-03-21T16:30:00+08:00         1.647778e+09
14  productcatalogservice2-0      container.node-6.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09
15        currencyservice2-0      container.node-6.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09
16         paymentservice2-0      container.node-6.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09
17  recommendationservice2-0      container.node-6.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09
18               frontend2-0      container.node-6.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09
19   productcatalogservice-2      container.node-5.container_start_time_seconds   1.647523e+09             30  2022-03-21T16:30:00+08:00         1.647523e+09}```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse existing variables if present
try:
    df2
except NameError:
    df2 = pd.read_csv('trace.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using ALL rows
quantiles = (
    df2.groupby(['cmdb_id', 'trace_name'])['value']
    .quantile([0.05, 0.95])
    .unstack(level=2)
    .reset_index()
)
# After unstack, columns may be [0.05,0.95] or named; normalize
quantiles.columns = ['cmdb_id', 'trace_name', 'p05', 'p95']

# 2) Convert timestamps to UTC+8 and filter to incident window (inclusive)
df2['ts'] = pd.to_datetime(df2['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df2_window = df2[(df2['ts'] >= start_ts) & (df2['ts'] <= end_ts)].copy()

# 3) For each row in window mark high/low anomalies relative to global thresholds
# Merge thresholds into windowed data
df2_window = df2_window.merge(quantiles, on=['cmdb_id', 'trace_name'], how='left')

# Mark high and low anomalies
df2_window['is_high'] = df2_window['value'] >= df2_window['p95']
df2_window['is_low'] = df2_window['value'] <= df2_window['p05']

# Aggregate counts and earliest timestamps and extremes per type
high_counts = (
    df2_window.groupby(['cmdb_id', 'trace_name'])['is_high']
    .sum()
    .reset_index(name='high_anomalies')
)
low_counts = (
    df2_window.groupby(['cmdb_id', 'trace_name'])['is_low']
    .sum()
    .reset_index(name='low_anomalies')
)

high_earliest = (
    df2_window[df2_window['is_high']]
    .groupby(['cmdb_id', 'trace_name'])['ts']
    .min()
    .reset_index(name='high_earliest_ts')
)
low_earliest = (
    df2_window[df2_window['is_low']]
    .groupby(['cmdb_id', 'trace_name'])['ts']
    .min()
    .reset_index(name='low_earliest_ts')
)

high_extreme = (
    df2_window[df2_window['is_high']]
    .groupby(['cmdb_id', 'trace_name'])['value']
    .max()
    .reset_index(name='high_extreme_value')
)
low_extreme = (
    df2_window[df2_window['is_low']]
    .groupby(['cmdb_id', 'trace_name'])['value']
    .min()
    .reset_index(name='low_extreme_value')
)

# Combine counts with full series list (quantiles) to include series without window rows
summary = quantiles.merge(high_counts, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(low_counts, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(high_earliest, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(low_earliest, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(high_extreme, on=['cmdb_id', 'trace_name'], how='left')
summary = summary.merge(low_extreme, on=['cmdb_id', 'trace_name'], how='left')

# Fill NaNs for counts with 0
summary['high_anomalies'] = summary['high_anomalies'].fillna(0).astype(int)
summary['low_anomalies'] = summary['low_anomalies'].fillna(0).astype(int)

# Determine predominant anomaly type (use 'high' when tie)
def pick_predominant(row):
    if row['high_anomalies'] >= row['low_anomalies'] and row['high_anomalies']>0:
        return 'high'
    elif row['low_anomalies'] > 0:
        return 'low'
    else:
        return None

summary['predominant_type'] = summary.apply(pick_predominant, axis=1)
# Determine anomaly_count, threshold_value_used, earliest timestamp, extreme value based on predominant_type
summary['anomaly_count'] = summary.apply(
    lambda r: r['high_anomalies'] if r['predominant_type']=='high' else (r['low_anomalies'] if r['predominant_type']=='low' else 0),
    axis=1
)
summary['threshold_value_used'] = summary.apply(
    lambda r: r['p95'] if r['predominant_type']=='high' else (r['p05'] if r['predominant_type']=='low' else None),
    axis=1
)
# earliest ts
summary['earliest_anomaly_ts'] = summary.apply(
    lambda r: r['high_earliest_ts'] if r['predominant_type']=='high' else (r['low_earliest_ts'] if r['predominant_type']=='low' else pd.NaT),
    axis=1
)
# extreme value: max for high, min for low
summary['extreme_value_in_window'] = summary.apply(
    lambda r: r['high_extreme_value'] if r['predominant_type']=='high' else (r['low_extreme_value'] if r['predominant_type']=='low' else None),
    axis=1
)

# Format earliest_anomaly_ts to ISO string with timezone colon, or None
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_ts'].apply(fmt_ts)

# Prepare final table columns
final = summary[[
    'cmdb_id',
    'trace_name',
    'predominant_type',
    'threshold_value_used',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'extreme_value_in_window'
]].copy()

# 4) Compact summary outputs
total_series_scanned = int(quantiles.shape[0])
# series with >=1 anomaly (either high or low) in window
series_with_any = int((final['anomaly_count'] > 0).sum())

# Top 20 by anomaly_count desc then by extreme magnitude (abs) desc for tie-breaker
top_20 = final.sort_values(['anomaly_count', 'extreme_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

result_summary = {
    'total_series_scanned': total_series_scanned,
    'series_with_>=1_anomaly_in_window': series_with_any,
    'top_20_by_anomaly_count': top_20
}

# Display the compact result (DataFrame limited to top 20)
result_summary

```
Out[3]:
```
```python
# Plain-English summary of the trace.csv anomaly analysis (uses result_summary from previous cell)
try:
    rs = result_summary
except NameError:
    raise RuntimeError("result_summary not found. Run the trace analysis cell first.")

total_series = rs['total_series_scanned']
series_with_any = rs['series_with_>=1_anomaly_in_window']
top20 = rs['top_20_by_anomaly_count']

# Build a concise human-readable summary string with representative top entries
examples = []
for i, row in top20.head(8).iterrows():
    examples.append(f"{row['cmdb_id']} : {row['trace_name']} ({row['predominant_type']}, count={int(row['anomaly_count'])})")

summary_text = (
    f"Trace.csv incident-window anomaly summary:\n\n"
    f"- Total distinct (cmdb_id, trace_name) series scanned: {total_series}\n"
    f"- Series with >=1 anomaly in the window (high or low relative to global P95/P5): {series_with_any}\n\n"
    f"- Key observations from the top-20 series by anomaly count:\n"
    f"  * All top entries show anomaly_count = 30 and earliest anomaly at 2022-03-21T16:30:00+08:00.\n"
    f"  * Predominant anomaly type for the top series is 'high' (value >= global P95), indicating elevated durations/error rates.\n"
    f"  * Representative top series include:\n    - " + "\n    - ".join(examples) + "\n\n"
    f"Interpretation (data-only): the incident window (2022-03-21T16:30:00+08:00 to 2022-03-21T17:00:00+08:00) "
    f"contains sustained high anomalies across many trace metrics (latencies and error rates). If you want the exact top-20 table, inspect rs['top_20_by_anomaly_count']."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_series_scanned': 1232, 'series_with_>=1_anomaly_in_window': 1232, 'top_20_by_anomaly_count':               cmdb_id                                  trace_name predominant_type  threshold_value_used  anomaly_count earliest_anomaly_timestamp  extreme_value_in_window
0       cartservice-2   trace.from_checkoutservice-2.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.046800
1   checkoutservice-2         trace.to_cartservice-2.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.046800
2       cartservice-2  trace.from_checkoutservice-2.duration_mean             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.017333
3   checkoutservice-2        trace.to_cartservice-2.duration_mean             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.017333
4          frontend-2                       trace.self.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.003067
5          frontend-1                       trace.self.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.002172
6          frontend-0                       trace.self.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.001637
7      cartservice2-0         trace.from_frontend2-0.duration_p95             high                 0.001             30  2022-03-21T16:30:00+08:00                 0.001000
8         frontend2-0        trace.to_cartservice2-0.duration_p95             high                 0.001             30  2022-03-21T16:30:00+08:00                 0.001000
9       cartservice-0          trace.from_frontend-1.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000900
10         frontend-1         trace.to_cartservice-0.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000900
11      cartservice-2          trace.from_frontend-1.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000700
12         frontend-1         trace.to_cartservice-2.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000700
13      cartservice-2          trace.from_frontend-0.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000050
14         frontend-0         trace.to_cartservice-2.duration_p95             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000050
15      cartservice-0                    trace.self.duration_mean             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000021
16        adservice-0            trace.from_frontend-0.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000000
17        adservice-0            trace.from_frontend-1.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000000
18        adservice-0            trace.from_frontend-2.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000000
19        adservice-1            trace.from_frontend-0.error_rate             high                 0.000             30  2022-03-21T16:30:00+08:00                 0.000000}```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import re

# Reuse existing variables if present
try:
    df3
except NameError:
    df3 = pd.read_csv('log.csv')
try:
    df4
except NameError:
    df4 = pd.read_csv('error_logs.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Incident window (inclusive) in UTC+8
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)

# --------------------
# Part 1: log.csv
# --------------------
# 1) Global P95 per (cmdb_id, log_name)
p95_logs = (
    df3.groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Convert timestamps and filter to window
df3['ts'] = pd.to_datetime(df3['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df3_window = df3[(df3['ts'] >= start_ts) & (df3['ts'] <= end_ts)].copy()

# 3) Merge thresholds, mark anomalies (value >= p95), aggregate per series
df3_window = df3_window.merge(p95_logs, on=['cmdb_id', 'log_name'], how='left')
df3_window['is_anomaly'] = df3_window['value'] >= df3_window['p95_threshold']

# Aggregations in-window
agg_counts = (
    df3_window.groupby(['cmdb_id', 'log_name'])
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)
earliest_anom = (
    df3_window[df3_window['is_anomaly']]
    .groupby(['cmdb_id', 'log_name'])['ts']
    .min()
    .reset_index(name='earliest_anomaly_ts')
)

# Combine with full series list to include series without window rows
log_summary_df = (
    p95_logs
    .merge(agg_counts, on=['cmdb_id', 'log_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'log_name'], how='left')
)

log_summary_df['anomaly_count'] = log_summary_df['anomaly_count'].fillna(0).astype(int)
# format earliest_anomaly_ts
def fmt_ts_or_none(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')
    return s[:-5] + s[-5:-2] + ':' + s[-2:]
log_summary_df['earliest_anomaly_timestamp'] = log_summary_df['earliest_anomaly_ts'].apply(fmt_ts_or_none)

# Keep required columns
log_summary_df = log_summary_df[[
    'cmdb_id','log_name','p95_threshold','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'
]]

# Top-20 by anomaly_count desc, then max_value_in_window desc
top_20_logs = log_summary_df.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

total_series_logs = int(p95_logs.shape[0])
series_with_anoms_logs = int((log_summary_df['anomaly_count'] > 0).sum())

# --------------------
# Part 2: error_logs.csv
# --------------------
# Convert timestamps to tz-aware
df4['ts'] = pd.to_datetime(df4['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

total_error_rows = int(df4.shape[0])

# Top 10 cmdb_id by error message count
top_10_error_cmdb = (
    df4['cmdb_id']
    .value_counts()
    .head(10)
    .rename_axis('cmdb_id')
    .reset_index(name='row_count')
)

# Filter window
df4_window = df4[(df4['ts'] >= start_ts) & (df4['ts'] <= end_ts)].copy()

# Earliest error timestamp in window (ISO)
if not df4_window.empty:
    earliest_error_ts_window = fmt_ts_or_none(df4_window['ts'].min())
else:
    earliest_error_ts_window = None

# For top 3 cmdb_ids (by overall error counts), return earliest 5 error entries within window
top3_ids = top_10_error_cmdb['cmdb_id'].head(3).tolist()
earliest_5_by_top3 = {}
for cid in top3_ids:
    sub = df4_window[df4_window['cmdb_id'] == cid].sort_values('ts').head(5).copy()
    if not sub.empty:
        sub['timestamp_iso'] = sub['ts'].apply(fmt_ts_or_none)
        earliest_5_by_top3[cid] = sub[['timestamp_iso','cmdb_id','message']].rename(columns={'timestamp_iso':'timestamp'}).reset_index(drop=True)
    else:
        earliest_5_by_top3[cid] = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Search messages in-window for keywords
keywords = ['OOM','Killed','Exception','restart','terminated','container started']
pattern = r'(' + r'|'.join([re.escape(k) for k in keywords]) + r')'
matches = df4_window[df4_window['message'].str.contains(pattern, case=False, na=False)].copy()
# Format timestamps
if not matches.empty:
    matches['timestamp'] = matches['ts'].apply(fmt_ts_or_none)
    matches = matches[['timestamp','cmdb_id','message']].reset_index(drop=True)
    # limit to top 50 matches for compactness
    matches_limited = matches.head(50)
else:
    matches_limited = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Assemble final compact dict-like result
result = {
    'log_summary': {
        'total_series_scanned': total_series_logs,
        'series_with_>=1_anomaly_in_window': series_with_anoms_logs,
        'top_20_by_anomaly_count': top_20_logs  # DataFrame (limited to 20 rows)
    },
    'error_logs_summary': {
        'total_rows': total_error_rows,
        'top_10_cmdb_id_by_error_count': top_10_error_cmdb,  # DataFrame
        'earliest_5_errors_for_top3_cmdb_ids_within_window': earliest_5_by_top3,  # dict of DataFrames (each up to 5 rows)
        'earliest_error_timestamp_in_window': earliest_error_ts_window,
        'messages_in_window_matching_keywords_limited_to_50': matches_limited  # DataFrame
    }
}

# Display the compact result (IPython will show DataFrames compactly)
result

```
Out[4]:
```
```python
# Plain-English summary string based on the log.csv and error_logs.csv analysis results
summary_text = (
    "Summary (plain English):\n\n"
    "1) log.csv analysis:\n"
    "- Total distinct (cmdb_id, log_name) series scanned: 62.\n"
    "- Series with >=1 anomaly in the incident window (value >= global P95): 62 (all scanned series).\n"
    "- Top anomalies (representative): many services (e.g., adservice-0/1/2, cartservice-*, checkoutservice-*, currencyservice-*, emailservice-*, paymentservice-*, productcatalogservice-*)\n"
    "  report log.error_count as the top log metric, each with anomaly_count = 30 and earliest anomaly timestamp = 2022-03-21T16:30:00+08:00 (max_value_in_window = 0.0 in these cases).\n\n"
    "Interpretation: every log series exceeded or equaled its global P95 during the 16:30–17:00 window; many top entries are log.error_count with repeated anomalies starting at 16:30:00+08:00.\n\n"
    "2) error_logs.csv analysis:\n"
    "- Total error log rows: 1,697.\n"
    "- Top 10 cmdb_id by error count (top 3 shown):\n"
    "    1. frontend-0 — 772 errors\n"
    "    2. frontend-1 — 535 errors\n"
    "    3. frontend-2 — 390 errors\n"
    "- Earliest error timestamp in the incident window: 2022-03-21T16:30:01+08:00.\n"
    "- For the top three cmdb_ids, the earliest 5 error entries within the window are (timestamps in UTC+8):\n"
    "    * frontend-0: 2022-03-21T16:30:02, 16:30:03, 16:30:06, 16:30:06, 16:30:10 — messages: 'severity: warning, message: failed to retrieve...'\n"
    "    * frontend-1: 2022-03-21T16:30:03, 16:30:07, 16:30:08, 16:30:13, 16:30:14 — messages: 'severity: warning, message: failed to retrieve...'\n"
    "    * frontend-2: 2022-03-21T16:30:01, 16:30:07, 16:30:11, 16:30:15, 16:30:19 — messages: 'severity: warning, message: failed to retrieve...'\n"
    "- Keyword search in windowed messages for 'OOM', 'Killed', 'Exception', 'restart', 'terminated', or 'container started' returned no matches (none found in the incident window).\n\n"
    "Overall takeaway: during 2022-03-21T16:30:00+08:00 to 2022-03-21T17:00:00+08:00 there are widespread log anomalies (log.error_count) across many services starting at 16:30, and high volumes of warning/error messages concentrated on frontend-0/1/2. No explicit OOM/Killed/Exception/restart/terminated/container-started keywords were detected in the window.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'log_summary': {'total_series_scanned': 62, 'series_with_>=1_anomaly_in_window': 62, 'top_20_by_anomaly_count':                     cmdb_id         log_name  p95_threshold  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0               adservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
1               adservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
2               adservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
3             cartservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
4             cartservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
5             cartservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
6         checkoutservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
7         checkoutservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
8         checkoutservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
9         currencyservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
10        currencyservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
11        currencyservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
12           emailservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
13           emailservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
14           emailservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
15         paymentservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
16         paymentservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
17         paymentservice-2  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
18  productcatalogservice-0  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0
19  productcatalogservice-1  log.error_count            0.0             30  2022-03-21T16:30:00+08:00                  0.0}, 'error_logs_summary': {'total_rows': 1697, 'top_10_cmdb_id_by_error_count':       cmdb_id  row_count
0  frontend-0        772
1  frontend-1        535
2  frontend-2        390, 'earliest_5_errors_for_top3_cmdb_ids_within_window': {'frontend-0':                    timestamp     cmdb_id                                            message
0  2022-03-21T16:30:02+08:00  frontend-0  severity: warning, message: failed to retrieve...
1  2022-03-21T16:30:03+08:00  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21T16:30:06+08:00  frontend-0  severity: warning, message: failed to retrieve...
3  2022-03-21T16:30:06+08:00  frontend-0  severity: warning, message: failed to retrieve...
4  2022-03-21T16:30:10+08:00  frontend-0  severity: warning, message: failed to retrieve..., 'frontend-1':                    timestamp     cmdb_id                                            message
0  2022-03-21T16:30:03+08:00  frontend-1  severity: warning, message: failed to retrieve...
1  2022-03-21T16:30:07+08:00  frontend-1  severity: warning, message: failed to retrieve...
2  2022-03-21T16:30:08+08:00  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21T16:30:13+08:00  frontend-1  severity: warning, message: failed to retrieve...
4  2022-03-21T16:30:14+08:00  frontend-1  severity: warning, message: failed to retrieve..., 'frontend-2':                    timestamp     cmdb_id                                            message
0  2022-03-21T16:30:01+08:00  frontend-2  severity: warning, message: failed to retrieve...
1  2022-03-21T16:30:07+08:00  frontend-2  severity: warning, message: failed to retrieve...
2  2022-03-21T16:30:11+08:00  frontend-2  severity: warning, message: failed to retrieve...
3  2022-03-21T16:30:15+08:00  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-21T16:30:19+08:00  frontend-2  severity: warning, message: failed to retrieve...}, 'earliest_error_timestamp_in_window': '2022-03-21T16:30:01+08:00', 'messages_in_window_matching_keywords_limited_to_50': Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: []}}```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing metric DataFrame 'df' if available; otherwise load it
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# Ensure timezone object for UTC+8
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column converted to timezone-aware datetimes (store as 'ts' to reuse)
df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Find kpi_name containing 'start_time' or 'StartTime' (case-insensitive)
mask_start_time = df['kpi_name'].str.contains('start_time', case=False, na=False)
df_start_time_all = df[mask_start_time].copy()

# 2) Filter those rows to the incident window (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df_start_time_window = df_start_time_all[(df_start_time_all['ts'] >= start_ts) & (df_start_time_all['ts'] <= end_ts)].copy()

# 3) Aggregate per cmdb_id across all start_time KPI variants
if not df_start_time_window.empty:
    agg = (
        df_start_time_window
        .groupby('cmdb_id')
        .agg(
            count_of_start_time_rows_in_window=('value', 'size'),
            earliest_metric_ts_in_window=('ts', 'min'),
            min_value_in_window=('value', 'min'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # Format earliest timestamp to ISO with timezone colon (+08:00)
    def fmt_ts(ts):
        if pd.isna(ts):
            return None
        s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
        return s[:-5] + s[-5:-2] + ':' + s[-2:]

    agg['earliest_metric_timestamp_in_window'] = agg['earliest_metric_ts_in_window'].apply(fmt_ts)
    # Reorder and keep requested column names
    agg = agg[[
        'cmdb_id',
        'count_of_start_time_rows_in_window',
        'earliest_metric_timestamp_in_window',
        'min_value_in_window',
        'max_value_in_window'
    ]]

    # 4) Build compact summary: total distinct cmdb_id and top 20 by count
    total_distinct_cmdb = int(agg.shape[0])
    top_20 = agg.sort_values('count_of_start_time_rows_in_window', ascending=False).head(20).reset_index(drop=True)
else:
    # No rows in window
    agg = pd.DataFrame(columns=[
        'cmdb_id',
        'count_of_start_time_rows_in_window',
        'earliest_metric_timestamp_in_window',
        'min_value_in_window',
        'max_value_in_window'
    ])
    total_distinct_cmdb = 0
    top_20 = agg

# Final compact JSON/dict-like result
result = {
    'total_distinct_cmdb_with_start_time_in_window': total_distinct_cmdb,
    'top_20_cmdb_by_start_time_row_count': top_20  # DataFrame (limited to top 20 rows)
}

result

```
Out[5]:
```
```python
# Plain-English summary of the start_time KPI analysis (uses 'result' from previous cell)
try:
    result
except NameError:
    raise RuntimeError("Expected 'result' variable from prior analysis not found. Please run the start_time analysis cell first.")

top20 = result['top_20_cmdb_by_start_time_row_count']
total_cnt = result['total_distinct_cmdb_with_start_time_in_window']

# Build a few representative examples from the top-20 table
examples = []
for _, r in top20.head(10).iterrows():
    cm = r['cmdb_id']
    mn = r['min_value_in_window']
    mx = r['max_value_in_window']
    # format numeric values as integer if they represent epoch seconds, else as-is
    try:
        mn_i = int(mn)
        mx_i = int(mx)
        examples.append(f"{cm}: min/max = {mn_i} / {mx_i}")
    except Exception:
        examples.append(f"{cm}: min/max = {mn} / {mx}")

summary_text = (
    f"Start-time KPI summary (metric.csv) for 2022-03-21T16:30:00+08:00 through 2022-03-21T17:00:00+08:00:\n\n"
    f"- Total distinct cmdb_id with 'start_time' metrics in-window: {total_cnt}\n"
    f"- Top cmdb_ids by count of start_time rows (top 20 shown):\n"
    f"  * Most top entries have 30 rows in-window and earliest metric timestamp = 2022-03-21T16:30:00+08:00.\n"
    f"  * Representative examples (cmdb_id : min_value / max_value in-window):\n    - " + "\n    - ".join(examples) + "\n\n"
    f"- Observations:\n"
    f"  * For the top entries, min_value_in_window == max_value_in_window (constant value across the window),\n"
    f"    indicating the reported start_time metric value did not change during the incident window for those cmdb_ids.\n\n"
    f"If you want the exact top-20 table (cmdb_id, count, earliest_metric_timestamp, min_value, max_value), inspect result['top_20_cmdb_by_start_time_row_count']."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_distinct_cmdb_with_start_time_in_window': 40, 'top_20_cmdb_by_start_time_row_count':                cmdb_id  count_of_start_time_rows_in_window earliest_metric_timestamp_in_window  min_value_in_window  max_value_in_window
0          adservice-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
1          adservice-1                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
2          adservice-2                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
3         adservice2-0                                  30           2022-03-21T16:30:00+08:00         1.647822e+09         1.647822e+09
4        cartservice-0                                  30           2022-03-21T16:30:00+08:00         1.647778e+09         1.647778e+09
5        cartservice-1                                  30           2022-03-21T16:30:00+08:00         1.647778e+09         1.647778e+09
6        cartservice-2                                  30           2022-03-21T16:30:00+08:00         1.647778e+09         1.647778e+09
7       cartservice2-0                                  30           2022-03-21T16:30:00+08:00         1.647841e+09         1.647841e+09
8    checkoutservice-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
9    checkoutservice-1                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
10   checkoutservice-2                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
11  checkoutservice2-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
12   currencyservice-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
13   currencyservice-1                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
14   currencyservice-2                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
15  currencyservice2-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
16      emailservice-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
17      emailservice-1                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
18      emailservice-2                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09
19     emailservice2-0                                  30           2022-03-21T16:30:00+08:00         1.647523e+09         1.647523e+09}```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse metric DataFrame 'df' and timezone 'tz' if present; otherwise load/create them
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column as timezone-aware UTC+8 in 'ts'
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window boundaries (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)

# 1) Filter kpi_name containing 'start_time' (case-insensitive)
mask_start_time = df['kpi_name'].str.contains('start_time', case=False, na=False)
df_start_all = df[mask_start_time].copy()

# 2) Filter to the incident window (inclusive)
df_start_window = df_start_all[(df_start_all['ts'] >= start_ts) & (df_start_all['ts'] <= end_ts)].copy()

# Helper to format timezone-aware timestamp to ISO8601+08:00 with colon in offset
def fmt_ts_iso(t):
    if pd.isna(t):
        return None
    s = t.strftime('%Y-%m-%dT%H:%M:%S%z')  # +0800
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

# If no rows in window, prepare empty outputs
if df_start_window.empty:
    top20 = pd.DataFrame(columns=[
        'start_time_value',
        'count_of_distinct_cmdb_ids',
        'total_rows',
        'example_cmdb_ids'
    ])
    earliest_info = {
        'earliest_start_time_value': None,
        'distinct_cmdb_ids_reporting_earliest_value': 0,
        'earliest_start_time_value_ISO_UTC+8': None
    }
else:
    # 1) Compute distinct start_time metric values observed in-window and aggregations
    # Aggregate by exact numeric value in 'value'
    grouped = (
        df_start_window
        .groupby('value', as_index=False)
        .agg(
            count_of_distinct_cmdb_ids=('cmdb_id', lambda x: int(pd.Series(x).nunique())),
            total_rows=('cmdb_id', 'size'),
            # collect up to 10 distinct cmdb_ids as examples (preserve order of appearance)
            example_cmdb_ids=('cmdb_id', lambda x: list(pd.Series(x).unique()[:10]))
        )
        .rename(columns={'value': 'start_time_value'})
    )

    # 2) Sort by count_of_distinct_cmdb_ids desc, then total_rows desc, take top 20
    top20 = grouped.sort_values(['count_of_distinct_cmdb_ids', 'total_rows'], ascending=[False, False]).head(20).reset_index(drop=True)

    # 3) Find the single earliest start_time_value (smallest numeric value)
    earliest_value = grouped['start_time_value'].min()
    distinct_cmdbs_for_earliest = int(df_start_window[df_start_window['value'] == earliest_value]['cmdb_id'].nunique())
    # Convert earliest_value (epoch seconds) to ISO8601+08:00
    earliest_dt_iso = fmt_ts_iso(pd.to_datetime(earliest_value, unit='s', utc=True).tz_convert(tz))

    earliest_info = {
        'earliest_start_time_value': float(earliest_value),
        'distinct_cmdb_ids_reporting_earliest_value': distinct_cmdbs_for_earliest,
        'earliest_start_time_value_ISO_UTC+8': earliest_dt_iso
    }

# Assemble compact JSON/dict-like result
result = {
    'total_distinct_start_time_values_in_window': int(0 if df_start_window.empty else grouped.shape[0]),
    'top_20_start_time_values': top20,  # DataFrame limited to 20 rows
    'earliest_start_time_value_info': earliest_info
}

result

```
Out[6]:
```
```python
# Plain-English summary of the start_time value distribution (uses 'result' from prior computation)
try:
    result
except NameError:
    raise RuntimeError("Expected 'result' variable not found. Run the start_time value aggregation cell first.")

top20 = result['top_20_start_time_values']
total_distinct_values = int(result['total_distinct_start_time_values_in_window'])
earliest_info = result['earliest_start_time_value_info']

# Build a few representative lines from top20 (show up to 6 entries with short example lists)
rows_to_show = min(6, len(top20))
examples_lines = []
for i in range(rows_to_show):
    r = top20.iloc[i]
    # start_time_value may be float in scientific notation; show as integer epoch
    try:
        epoch_int = int(float(r['start_time_value']))
    except Exception:
        epoch_int = r['start_time_value']
    cmdb_examples = r['example_cmdb_ids']
    # format example list as up to 5 ids
    ex_display = ', '.join(cmdb_examples[:5]) if isinstance(cmdb_examples, (list, tuple)) else str(cmdb_examples)
    examples_lines.append(f"- value {epoch_int} (reported by {int(r['count_of_distinct_cmdb_ids'])} distinct cmdb_ids, {int(r['total_rows'])} rows). Examples: {ex_display}")

summary_text = (
    f"Start_time value distribution (metric.csv) within 2022-03-21T16:30:00+08:00 to 2022-03-21T17:00:00+08:00:\n\n"
    f"- Total distinct start_time values observed in-window: {total_distinct_values}\n\n"
    f"- Top start_time values (by number of distinct cmdb_ids reporting that value) — top {rows_to_show} examples:\n"
    + "\n".join(examples_lines) + "\n\n"
    f"- Earliest start_time value (smallest numeric): {int(earliest_info['earliest_start_time_value'])} "
    f"(reported by {earliest_info['distinct_cmdb_ids_reporting_earliest_value']} distinct cmdb_id(s)); "
    f"which corresponds to {earliest_info['earliest_start_time_value_ISO_UTC+8']} (UTC+8).\n\n"
    f"Notes: Example lists are truncated to keep the summary compact. If you need the full top-20 table or full example lists, request the DataFrame display."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_distinct_start_time_values_in_window': 21, 'top_20_start_time_values':     start_time_value  count_of_distinct_cmdb_ids  total_rows                                   example_cmdb_ids
0       1.647523e+09                           7         210  [adservice-0, checkoutservice2-0, emailservice...
1       1.647523e+09                           4         120  [adservice-1, currencyservice-1, emailservice-...
2       1.647523e+09                           4         120  [checkoutservice-1, frontend-1, recommendation...
3       1.647523e+09                           4         120  [adservice-2, currencyservice-2, emailservice-...
4       1.647523e+09                           2          60             [checkoutservice-0, currencyservice-0]
5       1.647523e+09                           2          60       [checkoutservice-2, recommendationservice-2]
6       1.647523e+09                           2          60                    [frontend-2, shippingservice-2]
7       1.647523e+09                           2          60            [currencyservice2-0, paymentservice2-0]
8       1.647523e+09                           1          30                          [recommendationservice-0]
9       1.647523e+09                           1          30                                [shippingservice-0]
10      1.647523e+09                           1          30                          [productcatalogservice-0]
11      1.647523e+09                           1          30                          [productcatalogservice-1]
12      1.647523e+09                           1          30                          [productcatalogservice-2]
13      1.647523e+09                           1          30                                      [frontend2-0]
14      1.647523e+09                           1          30                         [recommendationservice2-0]
15      1.647523e+09                           1          30                         [productcatalogservice2-0]
16      1.647778e+09                           1          30                                    [cartservice-2]
17      1.647778e+09                           1          30                                    [cartservice-1]
18      1.647778e+09                           1          30                                    [cartservice-0]
19      1.647822e+09                           1          30                                     [adservice2-0], 'earliest_start_time_value_info': {'earliest_start_time_value': 1647523078.0, 'distinct_cmdb_ids_reporting_earliest_value': 1, 'earliest_start_time_value_ISO_UTC+8': '2022-03-17T21:17:58+08:00'}}```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re

# Reuse existing loaded error_logs DataFrame 'df4' and timezone 'tz' if present
try:
    df4
except NameError:
    df4 = pd.read_csv('error_logs.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware timestamp column 'ts' in UTC+8
df4['ts'] = pd.to_datetime(df4['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window boundaries (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)

# Helper to format ts to ISO8601+08:00 string
def fmt_ts_iso_col(s):
    if pd.isna(s):
        return None
    ss = s.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
    return ss[:-5] + ss[-5:-2] + ':' + ss[-2:]

# Task 1: Filter to window and specific frontends
frontend_ids = ["frontend-0", "frontend-1", "frontend-2"]
df4_window = df4[(df4['ts'] >= start_ts) & (df4['ts'] <= end_ts)].copy()
df4_frontends_window = df4_window[df4_window['cmdb_id'].isin(frontend_ids)].sort_values('ts').head(50).copy()

# Format timestamp column for output
df4_frontends_window['timestamp'] = df4_frontends_window['ts'].apply(fmt_ts_iso_col)
task1_output = df4_frontends_window[['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)

# Task 2: Keyword search across entire file (no time filter)
keywords = ['OOM', 'Killed', 'Exception', 'restart', 'terminated', 'container started']

# Build regex pattern for any keyword (case-insensitive)
# Use word-insensitive matching for phrases; escape keywords appropriately
pattern_any = r'(' + r'|'.join([re.escape(k) for k in keywords]) + r')'

# Find rows matching any keyword (case-insensitive)
matches_any = df4[df4['message'].str.contains(pattern_any, case=False, na=False)].copy().sort_values('ts').reset_index(drop=True)
# Limit returned matches to up to 20 rows
task2_matches_limited = matches_any.head(20).copy()
task2_matches_limited['timestamp'] = task2_matches_limited['ts'].apply(fmt_ts_iso_col)
task2_matches_limited = task2_matches_limited[['timestamp', 'cmdb_id', 'message']]

# Also compute counts per keyword across entire file (case-insensitive)
keyword_counts = []
for k in keywords:
    cnt = int(df4['message'].str.contains(re.escape(k), case=False, na=False).sum())
    keyword_counts.append({'keyword': k, 'match_count': cnt})
keyword_counts_df = pd.DataFrame(keyword_counts)

# Assemble compact JSON/dict-like result (DataFrames are kept compact)
result = {
    'frontends_window_earliest_up_to_50': task1_output,            # DataFrame up to 50 rows: timestamp, cmdb_id, message
    'keyword_search_matches_up_to_20': task2_matches_limited,     # DataFrame up to 20 rows: timestamp, cmdb_id, message
    'keyword_match_counts': keyword_counts_df                     # DataFrame: keyword, match_count
}

result

```
Out[7]:
```
```python
# Plain-English summary based on the previous error_logs analysis results (uses 'result' dict)
try:
    result
except NameError:
    raise RuntimeError("Expected 'result' not found. Run the error_logs analysis cell first.")

front_df = result['frontends_window_earliest_up_to_50']
kw_matches_df = result['keyword_search_matches_up_to_20']
kw_counts_df = result['keyword_match_counts']

# Extract compact facts
front_rows = int(front_df.shape[0])
earliest_front_ts = front_df['timestamp'].iloc[0] if front_rows > 0 else None
# sample message contents (unique, up to 3)
sample_messages = front_df['message'].unique().tolist()[:3]

# Keyword totals
kw_counts = {row['keyword']: int(row['match_count']) for _, row in kw_counts_df.iterrows()}

summary_text = (
    "Error logs investigation (plain English):\n\n"
    "1) Frontend error rows in the incident window (2022-03-21T16:30:00+08:00 through 2022-03-21T17:00:00+08:00):\n"
    f"- Returned up to {front_rows} earliest rows for frontend-0, frontend-1, frontend-2 (we returned 50 rows as requested).\n"
    f"- Earliest returned timestamp among these frontends: {earliest_front_ts}.\n"
    f"- Message content is dominated by warnings like: {sample_messages} (many rows show 'severity: warning, message: failed to retrieve...').\n\n"
    "2) Keyword search across all error logs (no time filter) for: OOM, Killed, Exception, restart, terminated, container started:\n"
    f"- Matches returned (limited to 20 rows): {int(kw_matches_df.shape[0])} rows (none returned in the limited sample).\n"
    "- Totals per keyword across the full error_logs.csv:\n"
    + "\n".join([f"  * {k}: {v}" for k, v in kw_counts.items()]) + "\n\n"
    "Interpretation:\n"
    "- The frontends produced many warning messages starting at ~2022-03-21T16:30:01+08:00; these messages are primarily 'failed to retrieve...' warnings.\n"
    "- The global keyword scan found zero occurrences for the specific terms checked (OOM, Killed, Exception, restart, terminated, container started).\n\n"
    "If you want the actual rows, inspect result['frontends_window_earliest_up_to_50'] or result['keyword_search_matches_up_to_20']."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'frontends_window_earliest_up_to_50':                     timestamp     cmdb_id                                            message
0   2022-03-21T16:30:01+08:00  frontend-2  severity: warning, message: failed to retrieve...
1   2022-03-21T16:30:02+08:00  frontend-0  severity: warning, message: failed to retrieve...
2   2022-03-21T16:30:03+08:00  frontend-0  severity: warning, message: failed to retrieve...
3   2022-03-21T16:30:03+08:00  frontend-1  severity: warning, message: failed to retrieve...
4   2022-03-21T16:30:06+08:00  frontend-0  severity: warning, message: failed to retrieve...
..                        ...         ...                                                ...
45  2022-03-21T16:30:45+08:00  frontend-0  severity: warning, message: failed to retrieve...
46  2022-03-21T16:30:47+08:00  frontend-0  severity: warning, message: failed to retrieve...
47  2022-03-21T16:30:49+08:00  frontend-1  severity: warning, message: failed to retrieve...
48  2022-03-21T16:30:49+08:00  frontend-2  severity: warning, message: failed to retrieve...
49  2022-03-21T16:30:50+08:00  frontend-0  severity: warning, message: failed to retrieve...

[50 rows x 3 columns], 'keyword_search_matches_up_to_20': Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 'keyword_match_counts':              keyword  match_count
0                OOM            0
1             Killed            0
2          Exception            0
3            restart            0
4         terminated            0
5  container started            0}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse metric DataFrame 'df' if available; otherwise load it
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# Ensure UTC+8 timezone object
tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column converted to timezone-aware datetimes in 'ts' (UTC+8)
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Restrict to node cmdb_ids
node_ids = ['node-1','node-2','node-3','node-4','node-5','node-6']
df_nodes_all = df[df['cmdb_id'].isin(node_ids)].copy()

# 1) Compute global P95 per (cmdb_id, kpi_name) using ALL available rows for those node-series
p95_nodes = (
    df_nodes_all
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Filter to the incident window (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df_nodes_window = df_nodes_all[(df_nodes_all['ts'] >= start_ts) & (df_nodes_all['ts'] <= end_ts)].copy()

# 3) Merge thresholds into windowed rows and mark anomalies (value >= global P95)
df_nodes_window = df_nodes_window.merge(p95_nodes, on=['cmdb_id','kpi_name'], how='left')
df_nodes_window['is_anomaly'] = df_nodes_window['value'] >= df_nodes_window['p95_threshold']

# Aggregate per series within window
if not df_nodes_window.empty:
    agg_window = (
        df_nodes_window.groupby(['cmdb_id','kpi_name'])
        .agg(
            anomaly_count=('is_anomaly','sum'),
            max_value_in_window=('value','max')
        )
        .reset_index()
    )
    earliest_anom = (
        df_nodes_window[df_nodes_window['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'])['ts']
        .min()
        .reset_index(name='earliest_anomaly_ts')
    )
else:
    agg_window = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','max_value_in_window'])
    earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_ts'])

# Combine with full p95 list to include series that had no rows in window
node_summary = (
    p95_nodes
    .merge(agg_window, on=['cmdb_id','kpi_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')
)

# Fill missing counts with 0
node_summary['anomaly_count'] = node_summary['anomaly_count'].fillna(0).astype(int)

# Format earliest_anomaly_ts to ISO8601+08:00 string or None
def fmt_ts_iso(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

node_summary['earliest_anomaly_timestamp'] = node_summary['earliest_anomaly_ts'].apply(fmt_ts_iso)

# Keep requested columns
node_summary_out = node_summary[[
    'cmdb_id','kpi_name','p95_threshold','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'
]]

# Compute totals and top-20
total_node_series_scanned = int(p95_nodes.shape[0])
num_node_series_with_anomalies = int((node_summary_out['anomaly_count'] > 0).sum())

top_20_node_series = node_summary_out.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Assemble compact result dict
node_result_summary = {
    'total_node_series_scanned': total_node_series_scanned,
    'node_series_with_>=1_anomaly_in_window': num_node_series_with_anomalies,
    'top_20_node_series_by_anomaly_count': top_20_node_series  # DataFrame limited to 20 rows
}

# Display the compact result (IPython will render DataFrame compactly)
node_result_summary

```
Out[8]:
```
```python
# Plain-English summary of the node-series anomaly analysis (metric.csv)
try:
    node_result_summary
except NameError:
    raise RuntimeError("node_result_summary not found. Run the node analysis cell first.")

top20 = node_result_summary['top_20_node_series_by_anomaly_count']
total = int(node_result_summary['total_node_series_scanned'])
with_anoms = int(node_result_summary['node_series_with_>=1_anomaly_in_window'])

# Build representative lines from top20 (top 6)
examples = []
for i in range(min(6, len(top20))):
    r = top20.iloc[i]
    cm = r['cmdb_id']
    kpi = r['kpi_name']
    p95 = r['p95_threshold']
    cnt = int(r['anomaly_count'])
    earliest = r['earliest_anomaly_timestamp']
    mx = r['max_value_in_window']
    examples.append(f"{cm} / {kpi}: anomalies={cnt}, p95={p95}, earliest={earliest}, max={mx}")

summary_text = (
    f"Node metrics anomaly summary (metric.csv) for 2022-03-21T16:30:00+08:00–2022-03-21T17:00:00+08:00:\n\n"
    f"- Total distinct (node, kpi_name) series scanned: {total}\n"
    f"- Node-series with >=1 high anomaly in-window (value >= global P95): {with_anoms}\n\n"
    f"- Key observations from the top node-series by anomaly count:\n"
    + "\n".join(f"  * {line}" for line in examples) + "\n\n"
    f"- Notes: The top entries are dominated by system.disk.total (all nodes) with anomaly_count = 30 and earliest anomaly at 2022-03-21T16:30:00+08:00. "
    f"Other recurring top KPIs include system.os.nofile.max and system.mem.total across nodes. These node-series show sustained values >= their historical P95 throughout the incident window.\n\n"
    f"If you need the exact top-20 table, inspect node_result_summary['top_20_node_series_by_anomaly_count']."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_node_series_scanned': 350, 'node_series_with_>=1_anomaly_in_window': 350, 'top_20_node_series_by_anomaly_count':    cmdb_id                kpi_name  p95_threshold  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0   node-4       system.disk.total   7.616300e+09             30  2022-03-21T16:30:00+08:00         7.616300e+09
1   node-1       system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
2   node-2       system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
3   node-3       system.disk.total   5.867011e+09             30  2022-03-21T16:30:00+08:00         5.867011e+09
4   node-6       system.disk.total   3.617924e+09             30  2022-03-21T16:30:00+08:00         3.617924e+09
5   node-5       system.disk.total   6.191419e+08             30  2022-03-21T16:30:00+08:00         6.191419e+08
6   node-5  system.fs.inodes.total   3.932160e+07             30  2022-03-21T16:30:00+08:00         3.932160e+07
7   node-1    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
8   node-2    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
9   node-3    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
10  node-4    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
11  node-5    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
12  node-6    system.os.nofile.max   6.553600e+06             30  2022-03-21T16:30:00+08:00         6.553600e+06
13  node-1        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
14  node-2        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
15  node-3        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
16  node-4        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
17  node-5        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
18  node-6        system.mem.total   3.201100e+04             30  2022-03-21T16:30:00+08:00         3.201100e+04
19  node-4         system.io.rkb_s   0.000000e+00             30  2022-03-21T16:30:00+08:00         3.400000e+01}```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing metric DataFrame 'df' and timezone 'tz' if present; otherwise load
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column as timezone-aware UTC+8 in 'ts'
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Identify candidate components: specified nodes + pod cmdb_ids present in metric.csv
node_ids = ['node-1','node-2','node-3','node-4','node-5','node-6']

# Allowed pod list from task description
allowed_pods = [
 'frontend-0','frontend-1','frontend-2','frontend2-0',
 'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0',
 'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
 'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
 'adservice-0','adservice-1','adservice-2','adservice2-0',
 'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
 'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
 'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
 'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
 'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0'
]

# Determine which allowed pods actually appear in metric.csv
pods_present = sorted(list(set(df['cmdb_id'].unique()) & set(allowed_pods)))

# Candidate components = nodes present in df (to be strict) + pods_present
nodes_present = sorted(list(set(df['cmdb_id'].unique()) & set(node_ids)))
candidates = nodes_present + pods_present

# 2) Compute global P95 per (cmdb_id, kpi_name) using ALL available rows (only for candidate components)
df_candidates_all = df[df['cmdb_id'].isin(candidates)].copy()

# Compute p95 thresholds (global, before any window filtering) per series
p95_series = (
    df_candidates_all
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 3) Filter to incident window (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df_window = df_candidates_all[(df_candidates_all['ts'] >= start_ts) & (df_candidates_all['ts'] <= end_ts)].copy()

# Merge p95 thresholds into windowed rows
df_window = df_window.merge(p95_series, on=['cmdb_id','kpi_name'], how='left')

# Mark high-anomaly rows where value >= p95 (use >= as instructed)
df_window['is_high_anomaly'] = df_window['value'] >= df_window['p95_threshold']

# For each series compute max_value_in_window and earliest anomaly ts (if any)
series_agg = (
    df_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        max_value_in_window=('value','max'),
        earliest_anomaly_ts=('ts', lambda x: x[df_window.loc[x.index, 'is_high_anomaly']].min() if 'is_high_anomaly' in df_window.columns else pd.NaT),
        any_anomaly=('is_high_anomaly','any')
    )
)

# Note: the lambda above relies on index alignment; to be safer compute earliest separately:
if series_agg['earliest_anomaly_ts'].isnull().any():
    # compute earliest anomaly per group directly
    earliest_per_series = df_window[df_window['is_high_anomaly']].groupby(['cmdb_id','kpi_name'])['ts'].min().reset_index(name='earliest_anomaly_ts')
    series_agg = series_agg.drop(columns=['earliest_anomaly_ts']).merge(earliest_per_series, on=['cmdb_id','kpi_name'], how='left')

# Merge p95 into series_agg to compute breach_ratio
series_agg = series_agg.merge(p95_series, on=['cmdb_id','kpi_name'], how='left')

# Calculate breach_ratio only for anomalous series (any_anomaly == True), with denominator guard
eps = 1e-9
series_agg['breach_ratio'] = np.where(
    series_agg['any_anomaly'],
    (series_agg['max_value_in_window'] - series_agg['p95_threshold']) / (series_agg['p95_threshold'].abs() + eps),
    0.0
)

# 4) Aggregate per cmdb_id (component)
# Keep only anomalous series for component-level aggregation
anomalous_series = series_agg[series_agg['any_anomaly']].copy()

# component-level aggregation
comp_agg = (
    anomalous_series
    .groupby('cmdb_id', as_index=False)
    .agg(
        max_breach_ratio=('breach_ratio','max'),
        count_of_anomalous_kpis=('kpi_name','nunique'),
        earliest_anomaly_ts=('earliest_anomaly_ts','min')
    )
)

# Determine component_level (node or pod)
def comp_level(cm):
    return 'node' if cm.startswith('node-') else 'pod'

comp_agg['component_level'] = comp_agg['cmdb_id'].apply(comp_level)

# Format earliest_anomaly_ts to ISO+08:00 string
def fmt_ts_iso(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # +0800
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

comp_agg['earliest_anomaly_timestamp'] = comp_agg['earliest_anomaly_ts'].apply(fmt_ts_iso)

# Reorder columns and sort by severity (max_breach_ratio desc)
comp_out = comp_agg[[
    'cmdb_id','component_level','max_breach_ratio','count_of_anomalous_kpis','earliest_anomaly_timestamp'
]].sort_values('max_breach_ratio', ascending=False).reset_index(drop=True)

# 5) Summary metrics
total_candidates_scanned = len(candidates)
components_with_anomaly = int(comp_out.shape[0])

top_20_components = comp_out.head(20)

# Assemble compact result
result = {
    'total_candidate_components_scanned': total_candidates_scanned,
    'components_with_>=1_anomalous_kpi_in_window': components_with_anomaly,
    'top_20_components_by_max_breach_ratio': top_20_components  # DataFrame limited to 20 rows
}

result

```
Out[9]:
```
```python
# Plain-English summary of component severity ranking from the previous analysis (uses 'result' variable)
try:
    result
except NameError:
    raise RuntimeError("Expected 'result' not found. Run the component severity analysis cell first.")

top20 = result['top_20_components_by_max_breach_ratio']
total_candidates = int(result['total_candidate_components_scanned'])
components_with_anoms = int(result['components_with_>=1_anomalous_kpi_in_window'])

# Build representative top entries (first 8)
lines = []
for i in range(min(8, len(top20))):
    r = top20.iloc[i]
    lines.append(
        f"{i+1}. {r['cmdb_id']} ({r['component_level']}): max_breach_ratio={r['max_breach_ratio']:.3g}, "
        f"anomalous_kpis={int(r['count_of_anomalous_kpis'])}, earliest={r['earliest_anomaly_timestamp']}"
    )

summary_text = (
    f"Anomaly severity summary (metric.csv) for candidate components in the incident window:\n\n"
    f"- Total candidate components scanned: {total_candidates}\n"
    f"- Components with >=1 anomalous KPI in-window: {components_with_anoms}\n\n"
    f"- Top components by max breach ratio (representative top entries):\n"
    + "\n".join(lines) + "\n\n"
    f"- Observations:\n"
    f"  * The highest-severity entries include pods (e.g., adservice-2 with an extremely large breach ratio) and node-4 (node) near the top.\n"
    f"  * Many components show anomalies across many KPIs (e.g., frontend-1 has 172 anomalous KPIs; cartservice pods show ~88–100 anomalous KPIs),\n"
    f"    and most earliest anomaly timestamps are 2022-03-21T16:30:00+08:00, indicating a widespread, simultaneous onset.\n"
    f"  * Very large breach_ratio values can be caused by metrics whose p95 is near zero (timestamps, counters, or small baselines), so interpret magnitude carefully — inspect the kpi_name details for those top components.\n\n"
    f"- Recommended next steps:\n"
    f"  1) For the top-ranked components (e.g., adservice-2, node-4, cartservice-1/2, frontend-1), examine the specific anomalous KPI names driving the high breach_ratio.\n"
    f"  2) Cross-check traces/logs for those components to distinguish real performance degradations from artifacts (e.g., timestamp/start_time metrics inflating ratios).\n\n"
    f"If you want, I can list the exact top-20 table of components or show the top anomalous KPI names for any selected component."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_candidate_components_scanned': 46, 'components_with_>=1_anomalous_kpi_in_window': 46, 'top_20_components_by_max_breach_ratio':               cmdb_id component_level  max_breach_ratio  count_of_anomalous_kpis earliest_anomaly_timestamp
0         adservice-2             pod      1.725000e+12                       90  2022-03-21T16:30:00+08:00
1              node-4            node      3.400000e+10                       59  2022-03-21T16:30:00+08:00
2       cartservice-1             pod      2.000000e+09                      100  2022-03-21T16:30:00+08:00
3       cartservice-2             pod      2.000000e+09                       88  2022-03-21T16:30:00+08:00
4   paymentservice2-0             pod      2.000000e+09                       70  2022-03-21T16:30:00+08:00
5         adservice-0             pod      2.000000e+09                       90  2022-03-21T16:30:00+08:00
6   currencyservice-1             pod      1.000000e+09                       76  2022-03-21T16:30:00+08:00
7   currencyservice-2             pod      1.000000e+09                       76  2022-03-21T16:30:00+08:00
8      cartservice2-0             pod      1.000000e+09                       96  2022-03-21T16:30:00+08:00
9   currencyservice-0             pod      1.000000e+09                       76  2022-03-21T16:30:00+08:00
10         frontend-1             pod      5.000000e+08                      172  2022-03-21T16:30:00+08:00
11             node-3            node      1.000000e+07                       58  2022-03-21T16:30:00+08:00
12     emailservice-2             pod      2.050000e+06                       74  2022-03-21T16:30:00+08:00
13  shippingservice-0             pod      1.050000e+06                       88  2022-03-21T16:30:00+08:00
14  shippingservice-2             pod      1.050000e+06                       80  2022-03-21T16:30:00+08:00
15             node-2            node      5.317273e+02                       59  2022-03-21T16:30:00+08:00
16  checkoutservice-0             pod      3.471425e+01                      134  2022-03-21T16:30:00+08:00
17             node-1            node      3.466966e+01                       58  2022-03-21T16:30:00+08:00
18   paymentservice-1             pod      1.684324e+01                       70  2022-03-21T16:30:00+08:00
19     emailservice-1             pod      1.509994e+01                       74  2022-03-21T16:30:00+08:00}```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing metric DataFrame 'df' and timezone 'tz' if available
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware timestamp column 'ts' (UTC+8)
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Retrieve the top-20 components table from prior result variable if present
# It should contain 'top_20_components_by_max_breach_ratio' as a DataFrame
if 'result' in globals() and isinstance(result, dict) and 'top_20_components_by_max_breach_ratio' in result:
    top_components_df = result['top_20_components_by_max_breach_ratio']
else:
    # fallback: try comp_out from prior cell
    top_components_df = globals().get('comp_out', pd.DataFrame())

# Take first 10 cmdb_ids (guard if fewer than 10)
components_examined = top_components_df['cmdb_id'].head(10).tolist()
total_components_examined = len(components_examined)

# Incident window boundaries (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)

# 1) Compute global P95 for all (cmdb_id, kpi_name) for these components using ALL available rows
p95_for_components = (
    df[df['cmdb_id'].isin(components_examined)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# 2) Filter metric rows to incident window and to these components
df_win_comp = df[
    (df['cmdb_id'].isin(components_examined)) &
    (df['ts'] >= start_ts) & (df['ts'] <= end_ts)
].copy()

# 3) Merge p95 and mark high anomalies (value >= p95)
df_win_comp = df_win_comp.merge(p95_for_components, on=['cmdb_id','kpi_name'], how='left')
df_win_comp['is_high_anomaly'] = df_win_comp['value'] >= df_win_comp['p95_threshold']

# 4) Aggregate per series: anomaly_count, max_value_in_window, earliest_anomaly_ts (only for anomaly rows)
# anomaly_count
series_counts = (
    df_win_comp.groupby(['cmdb_id','kpi_name'])['is_high_anomaly']
    .sum()
    .reset_index(name='anomaly_count_in_window')
)
# max value in window
series_max = (
    df_win_comp.groupby(['cmdb_id','kpi_name'])['value']
    .max()
    .reset_index(name='max_value_in_window')
)
# earliest anomaly timestamp per series (from anomaly rows)
anomaly_rows = df_win_comp[df_win_comp['is_high_anomaly']].copy()
earliest_anom = (
    anomaly_rows.groupby(['cmdb_id','kpi_name'])['ts']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_ts')
)

# Combine series-level info
series_summary = p95_for_components.merge(series_counts, on=['cmdb_id','kpi_name'], how='left')
series_summary = series_summary.merge(series_max, on=['cmdb_id','kpi_name'], how='left')
series_summary = series_summary.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for series with no window rows/anomalies
series_summary['anomaly_count_in_window'] = series_summary['anomaly_count_in_window'].fillna(0).astype(int)
# For max_value_in_window leave NaN if no rows in window

# 5) Compute breach_ratio for series that had anomaly_count>0
eps = 1e-9
series_summary['breach_ratio'] = np.where(
    series_summary['anomaly_count_in_window'] > 0,
    (series_summary['max_value_in_window'] - series_summary['p95_threshold']) / (series_summary['p95_threshold'].abs() + eps),
    0.0
)

# Format earliest anomaly timestamp to ISO+08:00 string
def fmt_ts_iso(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

series_summary['earliest_anomaly_timestamp_in_window'] = series_summary['earliest_anomaly_timestamp_ts'].apply(fmt_ts_iso)

# Keep only anomalous series (anomaly_count > 0)
anomalous_series = series_summary[series_summary['anomaly_count_in_window'] > 0].copy()

# 6) For each component, pick top 10 KPIs by breach_ratio desc
top_kpis_per_component = {}
for comp in components_examined:
    comp_series = anomalous_series[anomalous_series['cmdb_id'] == comp].copy()
    if comp_series.empty:
        top_kpis_per_component[comp] = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','p95_threshold','max_value_in_window','breach_ratio',
            'anomaly_count_in_window','earliest_anomaly_timestamp_in_window'
        ])
        continue
    comp_series = comp_series.sort_values('breach_ratio', ascending=False).head(10)
    # select requested columns and reset index
    comp_out = comp_series[[
        'cmdb_id','kpi_name','p95_threshold','max_value_in_window','breach_ratio',
        'anomaly_count_in_window','earliest_anomaly_timestamp_in_window'
    ]].reset_index(drop=True)
    top_kpis_per_component[comp] = comp_out

# Assemble final compact result
result_out = {
    'components_examined': components_examined,
    'total_components_examined': total_components_examined,
    'top_anomalous_kpis_per_component_limited_to_10': top_kpis_per_component  # mapping of cmdb_id -> DataFrame (up to 10 rows each)
}

# Display the compact result (DataFrames inside dict are bounded to requested sizes)
result_out

```
Out[10]:
```
```python
# Plain-English compact summary of the top anomalous KPIs for the selected 10 components.
# This cell reads the previously computed `result_out` and `top_anomalous_kpis_per_component_limited_to_10`
# and produces a concise human-readable summary string.

try:
    comps = result_out['components_examined']
    per_comp = result_out['top_anomalous_kpis_per_component_limited_to_10']
except Exception as e:
    raise RuntimeError("Expected analysis results not found in kernel. Please run the prior analysis cells.") from e

lines = []
lines.append(f"Components examined (count={len(comps)}): {', '.join(comps)}")
lines.append("")
lines.append("Summary of top anomalous KPIs (up to 10 KPIs per component, showing up to 3 representatives each):")
lines.append("")

# Helper to shorten KPI names for readability
def short(k, maxlen=70):
    s = k
    return (s if len(s) <= maxlen else s[:maxlen-3] + "...")

for comp in comps:
    dfk = per_comp.get(comp)
    if dfk is None or dfk.empty:
        lines.append(f"- {comp}: no anomalous KPIs in-window.")
        continue
    # take up to 3 representative KPIs per component for compactness
    reps = []
    for _, r in dfk.head(3).iterrows():
        kname = short(r['kpi_name'])
        br = r['breach_ratio']
        # format breach_ratio succinctly
        if abs(br) >= 1e6 or (br != 0 and abs(br) < 1e-2):
            br_s = f"{br:.3e}"
        else:
            br_s = f"{br:.3f}"
        earliest = r['earliest_anomaly_timestamp_in_window'] or "N/A"
        cnt = int(r['anomaly_count_in_window'])
        reps.append(f"{kname} (breach={br_s}, count={cnt}, earliest={earliest})")
    lines.append(f"- {comp}: " + "; ".join(reps))

lines.append("")
lines.append("Key takeaways:")
lines.append("- Many top breach_ratios are extremely large because the series' historical P95 is zero or near-zero (e.g., CPU throttling counters, some mesh/response counters). Such ratios should be interpreted cautiously and inspected by KPI name.")
lines.append("- Several pods show widespread anomalies across many KPIs (e.g., adservice-2, cartservice-1/2, paymentservice2-0, adservice-0, currencyservice-*).")
lines.append("- For many components the earliest anomalous timestamp is 2022-03-21T16:30:00+08:00, indicating a near-simultaneous onset across components.")
lines.append("- Recommended next step: for each top component, inspect the specific top KPIs listed (especially non-counter KPIs like memory_failures, fs_reads, network metrics) and cross-reference logs/traces for those components.")

summary_text = "\n".join(lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'components_examined': ['adservice-2', 'node-4', 'cartservice-1', 'cartservice-2', 'paymentservice2-0', 'adservice-0', 'currencyservice-1', 'currencyservice-2', 'cartservice2-0', 'currencyservice-0'], 'total_components_examined': 10, 'top_anomalous_kpis_per_component_limited_to_10': {'adservice-2':        cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  adservice-2  mesh.destination.frontend.adservice.istio_resp...       0.000000          1725.000000  1.725000e+12                       30            2022-03-21T16:30:00+08:00
1  adservice-2  mesh.destination.frontend.adservice.istio_requ...       0.000000           957.500000  9.575000e+11                       30            2022-03-21T16:30:00+08:00
2  adservice-2  mesh.destination.frontend.adservice.istio_requ...       0.000000             2.500000  2.500000e+09                       30            2022-03-21T16:30:00+08:00
3  adservice-2  container.node-5.container_memory_failures.hie...       0.000000             2.000000  2.000000e+09                       30            2022-03-21T16:30:00+08:00
4  adservice-2  container.node-5.container_memory_failures.con...       0.000000             2.000000  2.000000e+09                       30            2022-03-21T16:30:00+08:00
5  adservice-2  mesh.destination.frontend.adservice.istio_requ...       0.000000             0.752500  7.525000e+08                       30            2022-03-21T16:30:00+08:00
6  adservice-2  mesh.destination.frontend.adservice.istio_requ...       0.000000             0.500000  5.000000e+08                       30            2022-03-21T16:30:00+08:00
7  adservice-2  mesh.source.adservice.jaeger-collector.istio_r...       0.005464             0.789550  1.435070e+02                        2            2022-03-21T16:43:00+08:00
8  adservice-2    container.node-5.container_fs_reads_MB./dev/vda       0.001953             0.074219  3.699998e+01                        3            2022-03-21T16:30:00+08:00
9  adservice-2       container.node-5.container_fs_reads./dev/vda       0.500000            10.000000  1.900000e+01                        3            2022-03-21T16:30:00+08:00, 'node-4':   cmdb_id                     kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  node-4              system.io.rkb_s         0.0000                34.00  3.400000e+10                       30            2022-03-21T16:30:00+08:00
1  node-4                system.io.r_s         0.0000                 3.50  3.500000e+09                       30            2022-03-21T16:30:00+08:00
2  node-4            system.io.w_await         0.0000                 0.50  5.000000e+08                       30            2022-03-21T16:30:00+08:00
3  node-4            system.io.r_await         0.0000                 0.14  1.400000e+08                       30            2022-03-21T16:30:00+08:00
4  node-4           system.io.avg_q_sz         0.0000                 0.01  1.000000e+07                       30            2022-03-21T16:30:00+08:00
5  node-4              system.io.await         0.0770                 0.50  5.493506e+00                        2            2022-03-21T16:30:00+08:00
6  node-4                system.io.w_s         1.2750                 8.00  5.274510e+00                        2            2022-03-21T16:45:00+08:00
7  node-4              system.io.svctm         0.0165                 0.07  3.242424e+00                        2            2022-03-21T16:30:00+08:00
8  node-4            system.cpu.system         1.3855                 2.99  1.158066e+00                        2            2022-03-21T16:47:00+08:00
9  node-4  system.net.tcp.retrans_segs         0.2355                 0.48  1.038217e+00                        2            2022-03-21T16:45:00+08:00, 'cartservice-1':          cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  cartservice-1  container.node-5.container_cpu_cfs_throttled_p...       0.000000             2.000000  2.000000e+09                       30            2022-03-21T16:30:00+08:00
1  cartservice-1  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.050197  5.019655e+07                       30            2022-03-21T16:30:00+08:00
2  cartservice-1  container.node-5.container_network_receive_MB....       0.097036             0.621465  5.404493e+00                        2            2022-03-21T16:42:00+08:00
3  cartservice-1  container.node-5.container_memory_failures.hie...      82.300000           202.500000  1.460510e+00                        2            2022-03-21T16:53:00+08:00
4  cartservice-1  container.node-5.container_memory_failures.con...      82.300000           202.500000  1.460510e+00                        2            2022-03-21T16:53:00+08:00
5  cartservice-1         container.node-5.container_cpu_cfs_periods     259.425000           382.000000  4.724872e-01                        2            2022-03-21T16:39:00+08:00
6  cartservice-1       container.node-5.container_cpu_usage_seconds       0.613195             0.882357  4.389504e-01                        2            2022-03-21T16:53:00+08:00
7  cartservice-1        container.node-5.container_cpu_user_seconds       0.530250             0.760000  4.332862e-01                        2            2022-03-21T16:53:00+08:00
8  cartservice-1  container.node-5.container_network_transmit_MB...       1.295762             1.818579  4.034817e-01                        2            2022-03-21T16:49:00+08:00
9  cartservice-1      container.node-5.container_cpu_system_seconds       0.087750             0.100000  1.396011e-01                        2            2022-03-21T16:53:00+08:00, 'cartservice-2':          cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  cartservice-2  container.node-5.container_cpu_cfs_throttled_p...       0.000000             2.000000  2.000000e+09                       30            2022-03-21T16:30:00+08:00
1  cartservice-2  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.076278  7.627848e+07                       30            2022-03-21T16:30:00+08:00
2  cartservice-2  mesh.destination.checkoutservice.cartservice.i...       0.009791             0.060850  5.214732e+00                        2            2022-03-21T16:39:00+08:00
3  cartservice-2  container.node-5.container_memory_failures.con...      98.100000           602.000000  5.136595e+00                        2            2022-03-21T16:37:00+08:00
4  cartservice-2  container.node-5.container_memory_failures.hie...      98.100000           602.000000  5.136595e+00                        2            2022-03-21T16:37:00+08:00
5  cartservice-2  container.node-5.container_network_receive_MB....       0.106861             0.611115  4.718773e+00                        2            2022-03-21T16:50:00+08:00
6  cartservice-2        container.node-5.container_cpu_user_seconds       0.488250             0.720000  4.746544e-01                        2            2022-03-21T16:40:00+08:00
7  cartservice-2         container.node-5.container_cpu_cfs_periods     239.775000           344.000000  4.346783e-01                        2            2022-03-21T16:35:00+08:00
8  cartservice-2       container.node-5.container_cpu_usage_seconds       0.562875             0.806106  4.321224e-01                        2            2022-03-21T16:40:00+08:00
9  cartservice-2  container.node-5.container_network_transmit_MB...       0.826706             1.171831  4.174705e-01                        2            2022-03-21T16:50:00+08:00, 'paymentservice2-0':              cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  paymentservice2-0  container.node-6.container_cpu_cfs_throttled_p...   0.000000e+00             2.000000  2.000000e+09                       29            2022-03-21T16:30:00+08:00
1  paymentservice2-0  container.node-6.container_cpu_cfs_throttled_s...   3.552714e-15             0.141792  1.417912e+08                        8            2022-03-21T16:32:00+08:00
2  paymentservice2-0  container.node-6.container_network_receive_MB....   3.476219e-02             0.559597  1.509785e+01                        2            2022-03-21T16:34:00+08:00
3  paymentservice2-0        container.node-6.container_cpu_user_seconds   1.000000e-02             0.040000  3.000000e+00                        4            2022-03-21T16:33:00+08:00
4  paymentservice2-0       container.node-6.container_cpu_usage_seconds   1.283557e-02             0.050107  2.903757e+00                        2            2022-03-21T16:33:00+08:00
5  paymentservice2-0  container.node-6.container_memory_failures.con...   2.894000e+02           883.000000  2.051140e+00                        2            2022-03-21T16:33:00+08:00
6  paymentservice2-0  container.node-6.container_memory_failures.hie...   2.894000e+02           883.000000  2.051140e+00                        2            2022-03-21T16:33:00+08:00
7  paymentservice2-0         container.node-6.container_cpu_cfs_periods   1.020000e+01            18.000000  7.647059e-01                        2            2022-03-21T16:36:00+08:00
8  paymentservice2-0  container.node-6.container_network_transmit_MB...   2.888138e-01             0.392362  3.585281e-01                        2            2022-03-21T16:47:00+08:00
9  paymentservice2-0  container.node-6.container_network_transmit_pa...   2.638000e+02           339.666667  2.875916e-01                        2            2022-03-21T16:34:00+08:00, 'adservice-0':        cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  adservice-0  container.node-5.container_cpu_cfs_throttled_p...       0.000000             2.000000  2.000000e+09                       30            2022-03-21T16:30:00+08:00
1  adservice-0  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.063761  6.376098e+07                       30            2022-03-21T16:30:00+08:00
2  adservice-0  container.node-5.container_network_receive_MB....       0.086341             0.607247  6.033134e+00                        2            2022-03-21T16:35:00+08:00
3  adservice-0        container.node-5.container_cpu_user_seconds       0.075000             0.220000  1.933333e+00                        2            2022-03-21T16:45:00+08:00
4  adservice-0  mesh.source.adservice.jaeger-collector.istio_r...       0.005700             0.014100  1.473684e+00                        2            2022-03-21T16:47:00+08:00
5  adservice-0       container.node-5.container_cpu_usage_seconds       0.115056             0.251222  1.183470e+00                        2            2022-03-21T16:45:00+08:00
6  adservice-0  container.node-5.container_network_receive_pac...     634.850000           834.000000  3.136961e-01                        2            2022-03-21T16:35:00+08:00
7  adservice-0  container.node-5.container_network_transmit_pa...     460.850000           595.000000  2.910925e-01                        2            2022-03-21T16:35:00+08:00
8  adservice-0         container.node-5.container_cpu_cfs_periods     147.275000           186.000000  2.629435e-01                        2            2022-03-21T16:53:00+08:00
9  adservice-0  container.node-5.container_network_transmit_MB...       1.029758             1.281568  2.445332e-01                        2            2022-03-21T16:47:00+08:00, 'currencyservice-1':              cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  currencyservice-1  container.node-5.container_cpu_cfs_throttled_p...       0.000000             1.000000  1.000000e+09                       30            2022-03-21T16:30:00+08:00
1  currencyservice-1  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.039144  3.914430e+07                       30            2022-03-21T16:30:00+08:00
2  currencyservice-1  container.node-5.container_network_receive_MB....       0.174708             0.688100  2.938578e+00                        2            2022-03-21T16:49:00+08:00
3  currencyservice-1         container.node-5.container_cpu_cfs_periods     262.550000           326.000000  2.416683e-01                        2            2022-03-21T16:34:00+08:00
4  currencyservice-1        container.node-5.container_cpu_user_seconds       0.252750             0.310000  2.265084e-01                        2            2022-03-21T16:53:00+08:00
5  currencyservice-1       container.node-5.container_cpu_usage_seconds       0.309542             0.367339  1.867186e-01                        2            2022-03-21T16:34:00+08:00
6  currencyservice-1  mesh.destination.checkoutservice.currencyservi...       0.003675             0.004200  1.428571e-01                        2            2022-03-21T16:51:00+08:00
7  currencyservice-1      container.node-5.container_cpu_system_seconds       0.055000             0.060000  9.090909e-02                        2            2022-03-21T16:34:00+08:00
8  currencyservice-1  container.node-5.container_memory_failures.con...    3242.875000          3496.000000  7.805574e-02                        2            2022-03-21T16:34:00+08:00
9  currencyservice-1  container.node-5.container_memory_failures.hie...    3242.875000          3496.000000  7.805574e-02                        2            2022-03-21T16:34:00+08:00, 'currencyservice-2':              cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  currencyservice-2  container.node-5.container_cpu_cfs_throttled_p...       0.000000             1.000000  1.000000e+09                       30            2022-03-21T16:30:00+08:00
1  currencyservice-2  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.021034  2.103439e+07                       30            2022-03-21T16:30:00+08:00
2  currencyservice-2  container.node-5.container_network_receive_pac...     945.025000          1229.000000  3.004947e-01                        2            2022-03-21T16:32:00+08:00
3  currencyservice-2  container.node-5.container_network_transmit_MB...       0.863614             1.119378  2.961558e-01                        2            2022-03-21T16:30:00+08:00
4  currencyservice-2  container.node-5.container_network_transmit_pa...     915.175000          1184.000000  2.937416e-01                        2            2022-03-21T16:32:00+08:00
5  currencyservice-2  container.node-5.container_network_receive_MB....       0.326421             0.405301  2.416509e-01                        2            2022-03-21T16:55:00+08:00
6  currencyservice-2         container.node-5.container_cpu_cfs_periods     258.250000           304.000000  1.771539e-01                        2            2022-03-21T16:53:00+08:00
7  currencyservice-2  mesh.destination.checkoutservice.currencyservi...       0.002677             0.003150  1.764705e-01                        2            2022-03-21T16:35:00+08:00
8  currencyservice-2       container.node-5.container_cpu_usage_seconds       0.310306             0.337506  8.765701e-02                        2            2022-03-21T16:53:00+08:00
9  currencyservice-2  container.node-5.container_memory_failures.hie...    3266.725000          3526.000000  7.936848e-02                        2            2022-03-21T16:53:00+08:00, 'cartservice2-0':           cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  cartservice2-0  container.node-6.container_cpu_cfs_throttled_p...       0.000000             1.000000  1.000000e+09                       29            2022-03-21T16:30:00+08:00
1  cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...       0.000000             0.038993  3.899263e+07                       29            2022-03-21T16:30:00+08:00
2  cartservice2-0  container.node-6.container_network_receive_MB....       0.141342             0.628216  3.444641e+00                        2            2022-03-21T16:40:00+08:00
3  cartservice2-0      container.node-6.container_cpu_system_seconds       0.132333             0.205000  5.491184e-01                        2            2022-03-21T16:48:00+08:00
4  cartservice2-0       container.node-6.container_cpu_usage_seconds       0.558931             0.855192  5.300493e-01                        2            2022-03-21T16:48:00+08:00
5  cartservice2-0        container.node-6.container_cpu_user_seconds       0.404000             0.600000  4.851485e-01                        2            2022-03-21T16:48:00+08:00
6  cartservice2-0         container.node-6.container_cpu_cfs_periods     374.000000           553.000000  4.786096e-01                        2            2022-03-21T16:48:00+08:00
7  cartservice2-0  container.node-6.container_memory_failures.con...     131.200000           182.000000  3.871951e-01                        2            2022-03-21T16:50:00+08:00
8  cartservice2-0  container.node-6.container_memory_failures.hie...     131.200000           182.000000  3.871951e-01                        2            2022-03-21T16:50:00+08:00
9  cartservice2-0  container.node-6.container_network_receive_pac...     909.333333          1006.000000  1.063050e-01                        2            2022-03-21T16:47:00+08:00, 'currencyservice-0':              cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  breach_ratio  anomaly_count_in_window earliest_anomaly_timestamp_in_window
0  currencyservice-0  container.node-5.container_cpu_cfs_throttled_p...       0.000000             1.000000  1.000000e+09                       30            2022-03-21T16:30:00+08:00
1  currencyservice-0  container.node-5.container_cpu_cfs_throttled_s...       0.000000             0.022191  2.219068e+07                       30            2022-03-21T16:30:00+08:00
2  currencyservice-0  container.node-5.container_network_receive_MB....       0.179435             0.674149  2.757060e+00                        2            2022-03-21T16:30:00+08:00
3  currencyservice-0  mesh.destination.checkoutservice.currencyservi...       0.002389             0.003675  5.384613e-01                        2            2022-03-21T16:49:00+08:00
4  currencyservice-0         container.node-5.container_cpu_cfs_periods     250.125000           310.000000  2.393803e-01                        2            2022-03-21T16:50:00+08:00
5  currencyservice-0  container.node-5.container_network_transmit_MB...       0.801634             0.967735  2.072035e-01                        2            2022-03-21T16:53:00+08:00
6  currencyservice-0       container.node-5.container_cpu_usage_seconds       0.298103             0.330583  1.089584e-01                        2            2022-03-21T16:50:00+08:00
7  currencyservice-0      container.node-5.container_cpu_system_seconds       0.055500             0.060000  8.108108e-02                        2            2022-03-21T16:50:00+08:00
8  currencyservice-0  container.node-5.container_network_receive_pac...     917.100000           984.000000  7.294733e-02                        2            2022-03-21T16:53:00+08:00
9  currencyservice-0        container.node-5.container_cpu_user_seconds       0.241000             0.255000  5.809129e-02                        2            2022-03-21T16:50:00+08:00}}```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing trace DataFrame df2 if present, else load
try:
    df2
except NameError:
    df2 = pd.read_csv('trace.csv')

# Ensure timezone object for UTC+8
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp as timezone-aware UTC+8 in column 'ts'
if 'ts' not in df2.columns:
    df2['ts'] = pd.to_datetime(df2['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Select trace series whose trace_name includes the target substrings (case-insensitive)
pattern = r'(duration_mean|duration_p95|error_rate)'
mask = df2['trace_name'].str.contains(pattern, case=False, na=False)
df2_selected_all = df2[mask].copy()

# Compute total number of distinct (cmdb_id, trace_name) series scanned (using ALL rows)
p95_df = (
    df2_selected_all
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

total_series_scanned = int(p95_df.shape[0])

# 2) Filter trace.csv to the incident window (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df2_window = df2_selected_all[(df2_selected_all['ts'] >= start_ts) & (df2_selected_all['ts'] <= end_ts)].copy()

# 3) Merge global P95 into window rows and mark anomalies (value >= p95_threshold)
df2_window = df2_window.merge(p95_df, on=['cmdb_id','trace_name'], how='left')
df2_window['is_anomaly'] = df2_window['value'] >= df2_window['p95_threshold']

# Per-series aggregates within window
# anomaly_count, earliest anomaly ts (only where is_anomaly True), max value in window
series_agg_counts = (
    df2_window.groupby(['cmdb_id','trace_name'])
    .agg(
        anomaly_count=('is_anomaly','sum'),
        max_value_in_window=('value','max')
    )
    .reset_index()
)

# earliest anomaly timestamp per series (only from anomaly rows)
earliest_anom = (
    df2_window[df2_window['is_anomaly']]
    .groupby(['cmdb_id','trace_name'])['ts']
    .min()
    .reset_index(name='earliest_anomaly_ts')
)

# Combine series-level results with p95 (to ensure series with window rows are present)
series_summary = p95_df.merge(series_agg_counts, on=['cmdb_id','trace_name'], how='left')
series_summary = series_summary.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for series with no rows in window
series_summary['anomaly_count'] = series_summary['anomaly_count'].fillna(0).astype(int)
# max_value_in_window remains NaN where no rows in window

# 4) Aggregate per cmdb_id (receiver): sum anomaly_count and earliest anomaly among its series
# First, keep only series that had anomalies (anomaly_count > 0) for aggregation of earliest and sums
series_with_anom = series_summary[series_summary['anomaly_count'] > 0].copy()

# Sum anomaly counts per receiver
receiver_sum = (
    series_with_anom.groupby('cmdb_id', as_index=False)
    .agg(
        summed_anomaly_count=('anomaly_count','sum'),
        earliest_anomaly_ts=('earliest_anomaly_ts','min')
    )
)

# Also prepare example top trace_names per receiver: pick up to 5 trace_names with largest anomaly_count
top_trace_per_receiver = (
    series_with_anom
    .sort_values(['cmdb_id','anomaly_count'], ascending=[True, False])
    .groupby('cmdb_id')
    .apply(lambda g: g.sort_values('anomaly_count', ascending=False).head(5)['trace_name'].tolist())
    .reset_index(name='example_top_trace_names')
)

# Merge into receiver summary
receiver_summary = receiver_sum.merge(top_trace_per_receiver, on='cmdb_id', how='left')

# Format earliest_anomaly_ts to ISO8601+08:00 string
def fmt_ts_iso(ts):
    if pd.isna(ts):
        return None
    s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
    return s[:-5] + s[-5:-2] + ':' + s[-2:]

receiver_summary['earliest_anomaly_timestamp_among_series'] = receiver_summary['earliest_anomaly_ts'].apply(fmt_ts_iso)

# Keep only requested columns and sort by summed_anomaly_count desc
receiver_out = receiver_summary[[
    'cmdb_id',
    'summed_anomaly_count',
    'earliest_anomaly_timestamp_among_series',
    'example_top_trace_names'
]].sort_values('summed_anomaly_count', ascending=False).reset_index(drop=True)

# 5) Final compact summary
receivers_with_any_anomaly = int((receiver_out['summed_anomaly_count'] > 0).sum())
top_20_receivers = receiver_out.head(20)

result_summary = {
    'total_series_scanned': total_series_scanned,
    'receivers_with_>=1_anomaly_in_window': receivers_with_any_anomaly,
    'top_20_receivers_by_summed_anomaly_count': top_20_receivers  # DataFrame limited to 20 rows
}

# Display compact summary variable
result_summary

```
Out[11]:
```
```python
# Plain-English summary of trace receiver anomalies from the previous analysis
try:
    result_summary
except NameError:
    raise RuntimeError("Expected 'result_summary' not found. Please run the trace receiver analysis cell first.")

top20 = result_summary['top_20_receivers_by_summed_anomaly_count']
total_series = int(result_summary['total_series_scanned'])
receivers_with_anoms = int(result_summary['receivers_with_>=1_anomaly_in_window'])

# Build concise summary string
summary_lines = [
    f"Trace receiver anomaly summary (selected trace types: duration_mean, duration_p95, error_rate):",
    "",
    f"- Total distinct (cmdb_id, trace_name) series scanned (global P95 computed before window): {total_series}",
    f"- Receivers with >=1 anomaly in the incident window (value >= global P95): {receivers_with_anoms}",
    "",
    "Top receivers by summed anomaly count (representative top 10):",
]

# Add top 10 receivers with counts and earliest timestamps
for i, row in top20.head(10).iterrows():
    cm = row['cmdb_id']
    cnt = int(row['summed_anomaly_count'])
    earliest = row['earliest_anomaly_timestamp_among_series']
    examples = row['example_top_trace_names']
    # show up to 3 example trace names for compactness
    if isinstance(examples, (list, tuple)):
        ex = ", ".join(examples[:3])
    else:
        ex = str(examples)
    summary_lines.append(f"{i+1}. {cm}: summed_anomaly_count={cnt}, earliest_anomaly={earliest}, example_traces=[{ex}]")

summary_lines += [
    "",
    "Key observations:",
    "- The scan covered 924 series; 40 downstream receivers showed anomalies during 2022-03-21T16:30:00+08:00–17:00:00.",
    "- Top receivers are checkoutservice-2/1/0 and frontend-1/0/2 (highest summed anomaly counts), indicating widespread latency/error issues impacting many downstreams.",
    "- Example trace metrics driving anomalies are error_rate and duration metrics (duration_mean/duration_p95) from various sources (frontend, checkoutservice, etc.).",
    "- Earliest anomalies for top receivers mostly occur at 2022-03-21T16:30:00+08:00, suggesting a near-simultaneous onset across services.",
    "",
    "If you want the exact top-20 table or the full list of contributing trace series for any receiver, request the DataFrame 'top_20_receivers_by_summed_anomaly_count' or the per-receiver series detail."
]

summary_text = "\n".join(summary_lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_series_scanned': 924, 'receivers_with_>=1_anomaly_in_window': 40, 'top_20_receivers_by_summed_anomaly_count':                     cmdb_id  summed_anomaly_count earliest_anomaly_timestamp_among_series                            example_top_trace_names
0         checkoutservice-2                   926               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
1         checkoutservice-1                   924               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
2         checkoutservice-0                   912               2022-03-21T16:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
3                frontend-1                   833               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
4                frontend-0                   829               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
5                frontend-2                   778               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
6             cartservice-2                   490               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-0.duration_mean, t...
7             cartservice-0                   487               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-1.duration_mean, t...
8             cartservice-1                   432               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-0.duration_mean, t...
9   productcatalogservice-0                   341               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-1.error_rate, trac...
10  productcatalogservice-2                   341               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-1.error_rate, trac...
11  productcatalogservice-1                   341               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-0.error_rate, trac...
12              frontend2-0                   337               2022-03-21T16:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
13       checkoutservice2-0                   306               2022-03-21T16:30:00+08:00  [trace.from_frontend2-0.error_rate, trace.from...
14  recommendationservice-0                   241               2022-03-21T16:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
15  recommendationservice-2                   239               2022-03-21T16:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
16  recommendationservice-1                   239               2022-03-21T16:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
17        currencyservice-2                   208               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-0.error_rate, trac...
18        currencyservice-0                   206               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-0.error_rate, trac...
19        shippingservice-0                   205               2022-03-21T16:30:00+08:00  [trace.from_checkoutservice-1.error_rate, trac...}```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz
import re

# Reuse existing trace DataFrame if present, otherwise load
try:
    df2
except NameError:
    df2 = pd.read_csv('trace.csv')

# Reuse timezone if present, else create
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware timestamp column 'ts' in UTC+8
if 'ts' not in df2.columns:
    df2['ts'] = pd.to_datetime(df2['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Identify 'trace.to_' series (case-sensitive prefix) and compute global P95 per (cmdb_id, trace_name) using ALL rows
mask_to_series = df2['trace_name'].str.startswith('trace.to_', na=False)
df2_to_all = df2[mask_to_series].copy()

p95_to_series = (
    df2_to_all
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

total_to_series_scanned = int(p95_to_series.shape[0])

# 2) Filter trace.csv to the incident window (inclusive)
start_ts = pd.Timestamp('2022-03-21T16:30:00', tz=tz)
end_ts = pd.Timestamp('2022-03-21T17:00:00', tz=tz)
df2_window_to = df2_to_all[(df2_to_all['ts'] >= start_ts) & (df2_to_all['ts'] <= end_ts)].copy()

# 3) Merge global P95 into window rows and mark anomalous 'to' events (value >= p95)
df2_window_to = df2_window_to.merge(p95_to_series, on=['cmdb_id','trace_name'], how='left')
df2_window_to['is_anomalous_to'] = df2_window_to['value'] >= df2_window_to['p95_threshold']

# Keep only anomalous rows
anom_to_rows = df2_window_to[df2_window_to['is_anomalous_to']].copy()

# 4) Parse <target> from trace_name: substring between 'trace.to_' and next '.'
# Use regex capture; safe even if format varies
anom_to_rows['target'] = anom_to_rows['trace_name'].str.extract(r'^trace\.to_([^\.]+)\.', expand=False)

# Aggregate anomalies by target
if not anom_to_rows.empty:
    agg_by_target = (
        anom_to_rows.groupby('target')
        .agg(
            anomalous_event_count=('is_anomalous_to', 'sum'),
            earliest_anomalous_ts=('ts', 'min'),
            # collect up to 10 example source cmdb_ids preserving order of first appearance
            example_sources=('cmdb_id', lambda s: list(pd.Series(s).drop_duplicates().head(10)))
        )
        .reset_index()
    )
    # Format earliest timestamp to ISO8601+08:00 with colon
    def fmt_ts_iso(ts):
        if pd.isna(ts):
            return None
        s = ts.strftime('%Y-%m-%dT%H:%M:%S%z')  # e.g., +0800
        return s[:-5] + s[-5:-2] + ':' + s[-2:]
    agg_by_target['earliest_anomalous_timestamp_for_target'] = agg_by_target['earliest_anomalous_ts'].apply(fmt_ts_iso)
    # Keep requested columns and sort by anomalous_event_count desc
    agg_by_target_out = agg_by_target[[
        'target',
        'anomalous_event_count',
        'earliest_anomalous_timestamp_for_target',
        'example_sources'
    ]].sort_values('anomalous_event_count', ascending=False).reset_index(drop=True)
else:
    agg_by_target_out = pd.DataFrame(columns=[
        'target','anomalous_event_count','earliest_anomalous_timestamp_for_target','example_sources'
    ])

# 5) Build final compact summary (top 20 targets)
num_targets_with_anoms = int((agg_by_target_out['anomalous_event_count'] > 0).sum()) if not agg_by_target_out.empty else 0
top_20_targets = agg_by_target_out.head(20)

result = {
    'total_trace_to_series_scanned': total_to_series_scanned,
    'targets_with_>=1_anomalous_to_event': num_targets_with_anoms,
    'top_20_targets_by_anomalous_event_count': top_20_targets  # DataFrame limited to 20 rows
}

# Display the compact result (DataFrame rendered compactly by IPython)
result

```
Out[12]:
```
Summary (plain English):

- Scope and totals:
  - Total distinct "trace.to_" series scanned (global P95 computed using ALL rows): 568.
  - Targets with >=1 anomalous "to" event in the incident window (value >= series global P95): 36 targets.

- Top targets by anomalous "to" event count (top 20 shown; counts are anomalous rows during 2022-03-21T16:30:00+08:00—17:00:00):
  1. cartservice-2 — 452 anomalous events; earliest anomaly 2022-03-21T16:30:00+08:00; example sources: checkoutservice-0/1/2...
  2. cartservice-0 — 417 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  3. cartservice-1 — 394 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  4. productcatalogservice-1 — 387 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  5. productcatalogservice-0 — 385 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  6. productcatalogservice-2 — 384 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  7. shippingservice-2 — 242 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  8. shippingservice-0 — 240 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
  9. currencyservice-0 — 233 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 10. currencyservice-2 — 231 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 11. currencyservice-1 — 230 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 12. shippingservice-1 — 222 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 13. checkoutservice-0 — 149 anomalous events; earliest 16:30:00; example sources: frontend-0/1/2
 14. paymentservice-0 — 130 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 15. paymentservice-2 — 126 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/1/2...
 16. adservice-2 — 126 anomalous events; earliest 16:30:00; example sources: frontend-0/1/2
 17. emailservice-1 — 125 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/2/...
 18. emailservice-2 — 124 anomalous events; earliest 16:30:00; example sources: checkoutservice-0/2/...
 19. recommendationservice-1 — 123 anomalous events; earliest 16:30:00; example sources: frontend-0/1/2
 20. adservice-1 — 121 anomalous events; earliest 16:30:00; example sources: frontend-0/1/2

- Key observations and interpretation:
  - The highest counts are for cartservice and productcatalogservice targets (fed mainly by checkoutservice sources), indicating heavy anomalous "to" traffic/latency/error activity into these downstreams during the window.
  - Many targets show earliest anomaly at 2022-03-21T16:30:00+08:00 — anomalies started essentially simultaneously across many traces.
  - Example source lists show checkoutservice instances as frequent senders of anomalous "to" events to multiple downstreams; frontends also appear as sources for some targets (e.g., checkoutservice, adservice, recommendationservice).
  - Because global P95 was computed using all historical rows, these are relative high anomalies vs historical behavior; investigate the specific trace types (duration_p95, duration_mean, error_rate) contributing most anomalies per target to understand whether this is latency spikes, elevated error rates, or both.

- Suggested next steps:
  - For top targets (cartservice*, productcatalogservice*, shippingservice*, currencyservice*), inspect the contributing trace_names (duration vs error_rate) and timestamps in detail and correlate with logs/metrics for the identified source components (checkoutservice and frontends).
  - Prioritize targets with the largest anomalous_event_count and those where error_rate trace names are prominent, as these indicate functional failures rather than pure latency.

If you want, I can return the full top-20 table or show the specific trace_name breakdown for any selected target.

The original code execution output of IPython Kernel is also provided below for reference:

{'total_trace_to_series_scanned': 568, 'targets_with_>=1_anomalous_to_event': 36, 'top_20_targets_by_anomalous_event_count':                      target  anomalous_event_count earliest_anomalous_timestamp_for_target                                    example_sources
0             cartservice-2                    452               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
1             cartservice-0                    417               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
2             cartservice-1                    394               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
3   productcatalogservice-1                    387               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
4   productcatalogservice-0                    385               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
5   productcatalogservice-2                    384               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
6         shippingservice-2                    242               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
7         shippingservice-0                    240               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
8         currencyservice-0                    233               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
9         currencyservice-2                    231               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
10        currencyservice-1                    230               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
11        shippingservice-1                    222               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
12        checkoutservice-0                    149               2022-03-21T16:30:00+08:00               [frontend-0, frontend-1, frontend-2]
13         paymentservice-0                    130               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
14         paymentservice-2                    126               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-1, checkou...
15              adservice-2                    126               2022-03-21T16:30:00+08:00               [frontend-0, frontend-1, frontend-2]
16           emailservice-1                    125               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-2, checkou...
17           emailservice-2                    124               2022-03-21T16:30:00+08:00  [checkoutservice-0, checkoutservice-2, checkou...
18  recommendationservice-1                    123               2022-03-21T16:30:00+08:00               [frontend-0, frontend-1, frontend-2]
19              adservice-1                    121               2022-03-21T16:30:00+08:00               [frontend-0, frontend-1, frontend-2]}```
```